# Full pipeline (module source inlined)

First cell: seed 42, paths, `uv` kernel **sahel-malaria**.

Runtime independence: each module below is pasted in order (`config`, `io_dhs`, `inventory`, `load_merge`, `study_population`, `features`, `models_loco`, `shap_explain`, `sensitivity`, `figures_tables`). `from analysis.*` imports are stripped so names come from earlier cells. The `analysis/` package is **not deleted**; reviewers already received it via OSF.

This notebook **does not** overwrite `metrics_loco_primary.csv` or regenerate Figures 1–5. Primary tables must already exist from a previous `uv run python -m analysis.run_all`.


In [ ]:
from pathlib import Path
import os
import sys
import json
from dataclasses import dataclass

SEED = 42
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "analysis").is_dir() else HERE.parent
os.chdir(ROOT)
print("ROOT", ROOT)
print("SEED", SEED)
print("Use kernel sahel-malaria (uv .venv)")
print(sys.executable)


## Inlined `config.py`


In [ ]:
"""Pipeline configuration (plan file: 00_config.py)."""

from __future__ import annotations

from pathlib import Path

SEED = 42
N_BOOTSTRAP = 500
DPI = 300
RDT_POSITIVE_FLAG_N = 30
SUBGROUP_RDT_POS_FLAG_N = 50

ROOT = Path.cwd().resolve()  # notebook cwd is the repository root
DHS_ROOT = ROOT / "DHS_clean"
OUTPUTS = ROOT / "outputs"
TABLES = OUTPUTS / "tables"
FIGURES = OUTPUTS / "figures"
DATA_DIR = OUTPUTS / "data"
MODELS_DIR = OUTPUTS / "models"
REPORTS = ROOT / "reports"

COUNTRY_DIR_TO_NAME = {
    "Burkina_Faso": "Burkina Faso",
    "Gambia": "Gambia",
    "Mali": "Mali",
    "Mauritania": "Mauritania",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Senegal": "Senegal",
}

# Surveys that cannot support the afebrile endpoint (no KR / no fever item).
EXCLUDE_NO_KR = {("Mali", "2010")}

# Recode collisions: same country-year restated in two folders. Filled after inventory
# if two stems share a modal interview year; prefer the dedicated year folder.
PREFERRED_STEMS: dict[tuple[str, str], str] = {}

KR_CORE = [
    "caseid",
    "v000",
    "v001",
    "v002",
    "v003",
    "v005",
    "v006",
    "v007",
    "v008",
    "v016",
    "v021",
    "v022",
    "v023",
    "v024",
    "v025",
    "v106",
    "v113",
    "v116",
    "v136",
    "v190",
    "b4",
    "b16",
    "b19",
    "hw1",
    "h22",
    "h32y",
    "h32z",
    "hw53",
    "hml32",
    "hml35",
]

KR_OPTIONAL_PREFIXES = ("ml13", "h37")

HR_HH = [
    "hhid",
    "hv001",
    "hv002",
    "hv005",
    "hv006",
    "hv007",
    "hv021",
    "hv022",
    "hv023",
    "hv024",
    "hv025",
    "hv009",
    "hv213",
    "hv214",
    "hv215",
    "hv253",
    "hml1",
]

HR_PERSON_PREFIXES = ("hml35", "hml32", "hml12", "hml16", "sh212", "sh214r")

# JMP-style improved water / sanitation (DHS recode groups).
IMPROVED_WATER = {11, 12, 13, 14, 21, 31, 41, 51, 61, 62, 71}
IMPROVED_TOILET = {11, 12, 13, 14, 15, 21, 22, 41}

# hml12: 1 treated net, 2 untreated, 3 both.
NET_USED_CODES = {1, 2, 3}

# Standard hml35: 0 neg, 1 pos. Senegal 2008 sh212: 1 pos, 2 neg.
VALID_RDT_STANDARD = {0, 1}

DCA_THRESHOLDS = [i / 100 for i in range(5, 41)]
CALIBRATION_BINS = 10

LGBM_GRID = [
    {"num_leaves": 15, "learning_rate": 0.05, "n_estimators": 200, "max_depth": 3},
    {"num_leaves": 31, "learning_rate": 0.05, "n_estimators": 300, "max_depth": 5},
    {"num_leaves": 31, "learning_rate": 0.1, "n_estimators": 150, "max_depth": 4},
]

ELASTICNET_GRID = {"l1_ratio": [0.2, 0.5, 0.8], "C": [0.1, 1.0, 10.0]}

MICROSCOPY_HEADLINE = [
    ("Burkina Faso", "2021"),
    ("Nigeria", "2021"),
]

PRIMARY_FEATURES = [
    "wealth",
    "urban",
    "educ",
    "hhsize",
    "water_improved",
    "toilet_improved",
    "net_own",
    "net_used",
    "floor_grp",
    "wall_grp",
    "roof_grp",
    "age_months",
    "female",
    "month_sin",
    "month_cos",
    "lat_band",
    "sahel_nigeria",
]

HB_FEATURE = "hb_gdl"


def ensure_dirs() -> None:
    for path in (OUTPUTS, TABLES, FIGURES, DATA_DIR, MODELS_DIR, REPORTS):
        path.mkdir(parents=True, exist_ok=True)


## Inlined `util.py`


In [ ]:
"""Small helpers."""

from __future__ import annotations

import numpy as np
import pandas as pd


def df_to_md(df: pd.DataFrame, max_rows: int = 200) -> str:
    if df is None or df.empty:
        return "_empty_"
    view = df.head(max_rows)
    cols = [str(c) for c in view.columns]

    def cell(v) -> str:
        if v is None or (isinstance(v, float) and np.isnan(v)) or pd.isna(v):
            return ""
        if isinstance(v, float):
            return f"{v:.4g}"
        return str(v).replace("|", "/")

    lines = [
        "| " + " | ".join(cols) + " |",
        "| " + " | ".join("---" for _ in cols) + " |",
    ]
    for _, row in view.iterrows():
        lines.append("| " + " | ".join(cell(v) for v in row) + " |")
    if len(df) > max_rows:
        lines.append(f"| ... | {len(df) - max_rows} more rows |")
    return "\n".join(lines)


def weighted_mean(y: np.ndarray, w: np.ndarray) -> float:
    w = np.asarray(w, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.where(np.isfinite(w) & (w > 0), w, 0.0)
    if w.sum() <= 0:
        return float(np.nanmean(y))
    return float(np.average(y, weights=w))


def nest_ids(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    year = out.get("survey_year", pd.Series("", index=out.index)).astype(str)
    country = out["country"].astype(str)
    psu_raw = out["v001"] if "v001" in out.columns else out.get("hv001")
    strat_raw = out["v022"] if "v022" in out.columns else pd.Series(np.nan, index=out.index)
    if "v023" in out.columns:
        strat_raw = strat_raw.fillna(out["v023"])
    out["psu"] = country + "|" + year + "|" + psu_raw.astype("Int64").astype(str)
    out["stratum"] = country + "|" + year + "|" + strat_raw.astype("Int64").astype(str)
    if "weight" not in out.columns:
        out["weight"] = pd.to_numeric(out.get("v005"), errors="coerce") / 1_000_000
    if "hv005" in out.columns:
        alt = pd.to_numeric(out["hv005"], errors="coerce") / 1_000_000
        out["weight"] = out["weight"].fillna(alt)
    out["weight"] = out["weight"].fillna(0).clip(lower=0)
    return out


## Inlined `io_dhs.py`


In [ ]:
"""DHS I/O helpers: DTA column census, HR person-reshape, GPS DBF read."""

from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
import pyreadstat


_PERSON_RE = re.compile(
    r"^(hml35|hml32|hml12|hml16|sh212|sh214r)_(\d+)$",
    re.IGNORECASE,
)
_STEM_RE = re.compile(r"^([A-Z]{2})(KR|HR|GE)(.+)FL$", re.IGNORECASE)


def parse_recode_stem(path: Path) -> tuple[str, str, str] | None:
    m = _STEM_RE.match(path.stem)
    if not m:
        return None
    return m.group(1).upper(), m.group(2).upper(), m.group(3).upper()


def list_dta_columns(path: Path) -> list[str]:
    _, meta = pyreadstat.read_dta(str(path), metadataonly=True)
    return list(meta.column_names)


def map_lower(names: list[str]) -> dict[str, str]:
    return {n.lower(): n for n in names}


def dta_labels_na(path: Path) -> dict[str, bool]:
    """True if the dictionary label starts with NA (variable not collected)."""
    _, meta = pyreadstat.read_dta(str(path), metadataonly=True)
    labels = getattr(meta, "column_labels", None) or []
    names = list(meta.column_names)
    out: dict[str, bool] = {}
    for name, lab in zip(names, labels):
        text = (lab or "").strip()
        out[name.lower()] = text.upper().startswith("NA") or text.upper().startswith("NA-")
    return out


def read_dta(path: Path, wanted: list[str] | None = None) -> pd.DataFrame:
    names = list_dta_columns(path)
    lower_map = map_lower(names)
    usecols = None
    if wanted is not None:
        usecols = [lower_map[w] for w in wanted if w in lower_map]
        if not usecols:
            return pd.DataFrame()
    try:
        df, _ = pyreadstat.read_dta(str(path), usecols=usecols)
    except UnicodeDecodeError:
        df, _ = pyreadstat.read_dta(str(path), usecols=usecols, encoding="latin1")
    df.columns = [c.lower() for c in df.columns]
    return df


def wanted_kr_columns(available_lower: list[str]) -> list[str]:
    keep = [c for c in KR_CORE if c in available_lower]
    for col in available_lower:
        if col.startswith(KR_OPTIONAL_PREFIXES) and col not in keep:
            keep.append(col)
    return keep


def wanted_hr_columns(available_lower: list[str]) -> list[str]:
    keep = [c for c in HR_HH if c in available_lower]
    for col in available_lower:
        if _PERSON_RE.match(col) and col not in keep:
            keep.append(col)
    return keep


def reshape_person(hr: pd.DataFrame) -> pd.DataFrame:
    """Wide HR person slots (hml35_01, ...) to long rows keyed by hvidx."""
    slot_map: dict[str, dict[int, str]] = {}
    for col in hr.columns:
        m = _PERSON_RE.match(col)
        if not m:
            continue
        prefix = m.group(1).lower()
        idx = int(m.group(2))
        slot_map.setdefault(prefix, {})[idx] = col
    if not slot_map:
        return pd.DataFrame(columns=["hv001", "hv002", "hvidx"])

    id_cols = [c for c in ("hv001", "hv002") if c in hr.columns]
    frames = []
    indices = sorted({i for d in slot_map.values() for i in d})
    for idx in indices:
        piece = hr[id_cols].copy()
        piece["hvidx"] = idx
        empty = True
        for prefix, mapping in slot_map.items():
            col = mapping.get(idx)
            if col is None:
                piece[prefix] = pd.NA
                continue
            piece[prefix] = hr[col].values
            if piece[prefix].notna().any():
                empty = False
        if not empty:
            frames.append(piece)
    if not frames:
        return pd.DataFrame(columns=id_cols + ["hvidx"])
    long = pd.concat(frames, ignore_index=True)
    value_cols = [c for c in long.columns if c not in id_cols + ["hvidx"]]
    if value_cols:
        long = long.loc[long[value_cols].notna().any(axis=1)].copy()
    return long


def recode_rdt(series: pd.Series, scheme: str) -> pd.Series:
    """Return 1/0/NA. scheme='standard' (0/1) or 'sh212' (1 pos, 2 neg)."""
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(pd.NA, index=s.index, dtype="Float64")
    if scheme == "sh212":
        out = out.mask(s == 1, 1.0)
        out = out.mask(s == 2, 0.0)
    else:
        out = out.mask(s == 1, 1.0)
        out = out.mask(s == 0, 0.0)
    return out


def read_gps_dbf(dbf_path: Path) -> pd.DataFrame:
    from dbfread import DBF

    try:
        recs = list(DBF(str(dbf_path), encoding="latin-1", load=True))
    except UnicodeDecodeError:
        recs = list(DBF(str(dbf_path), encoding="cp1252", load=True))
    if not recs:
        return pd.DataFrame()
    gps = pd.DataFrame(recs)
    gps.columns = [str(c).lower() for c in gps.columns]
    cluster_col = next(
        (c for c in ("dhsclust", "cluster", "hv001", "dhsid") if c in gps.columns),
        None,
    )
    if cluster_col is None:
        return pd.DataFrame()
    lat_col = next((c for c in ("latnum", "latitude", "lat") if c in gps.columns), None)
    lon_col = next((c for c in ("longnum", "longitude", "long", "lon") if c in gps.columns), None)
    out = pd.DataFrame({"hv001": pd.to_numeric(gps[cluster_col], errors="coerce")})
    if lat_col:
        out["gps_lat"] = pd.to_numeric(gps[lat_col], errors="coerce")
    if lon_col:
        out["gps_lon"] = pd.to_numeric(gps[lon_col], errors="coerce")
    out = out.dropna(subset=["hv001"])
    out["hv001"] = out["hv001"].astype(int)
    if "gps_lat" in out.columns:
        out.loc[out["gps_lat"].isin([0, 9999, -9999]), "gps_lat"] = pd.NA
    if "gps_lon" in out.columns:
        out.loc[out["gps_lon"].isin([0, 9999, -9999]), "gps_lon"] = pd.NA
    return out.drop_duplicates("hv001")


def antimalarial_any(kr: pd.DataFrame) -> pd.Series:
    """Any recent antimalarial among ml13* / h37* malaria-drug items."""
    # ml13a–h and h37a–h typically include antimalarials; skip symptom-relief letters when labelled NA.
    candidates = [
        c
        for c in kr.columns
        if re.match(r"^(ml13[a-h]|h37[a-h]|h37e|h37da|h37aa|h37ab)$", c)
    ]
    if not candidates:
        return pd.Series(pd.NA, index=kr.index, dtype="Float64")
    block = kr[candidates].apply(pd.to_numeric, errors="coerce")
    taken = (block == 1).any(axis=1)
    asked = block.notna().any(axis=1)
    out = pd.Series(pd.NA, index=kr.index, dtype="Float64")
    out = out.mask(asked, taken.astype(float))
    return out


## Inlined `inventory.py`


In [ ]:
"""Survey manifest and per-country-year variable dictionary (plan: 01_inventory.py)."""

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd



INTEREST_VARS = {
    "rdt": ["hml35", "sh212"],
    "microscopy": ["hml32"],
    "fever": ["h22"],
    "care_seeking": ["h32y", "h32z"],
    "age": ["b19", "hw1"],
    "child_line": ["b16"],
    "weight": ["v005", "hv005"],
    "psu": ["v001", "hv001"],
    "stratum": ["v022", "v023"],
    "wealth": ["v190"],
    "residence": ["v025"],
    "region": ["v024"],
    "education": ["v106"],
    "hhsize": ["v136", "hv009"],
    "water": ["v113"],
    "toilet": ["v116"],
    "nets_hh": ["hml1"],
    "net_use": ["hml12"],
    "irs": ["hv253"],
    "floor": ["hv213"],
    "wall": ["hv214"],
    "roof": ["hv215"],
    "sex": ["b4"],
    "month": ["v006", "hv006"],
    "haemoglobin": ["hw53"],
}


def _find_files(folder: Path, kind: str) -> list[Path]:
    if not folder.is_dir():
        return []
    hits = []
    for p in folder.iterdir():
        if p.suffix.upper() != ".DTA" and not (kind == "GPS" and p.suffix.lower() == ".dbf"):
            continue
        if kind == "GPS":
            if p.suffix.lower() == ".dbf" and "GE" in p.stem.upper():
                hits.append(p)
        else:
            parsed = parse_recode_stem(p)
            if parsed and parsed[1] == ("KR" if kind == "KR" else "HR"):
                hits.append(p)
    return sorted(hits)


def discover_surveys() -> pd.DataFrame:
    rows = []
    for country_dir, country in COUNTRY_DIR_TO_NAME.items():
        cpath = DHS_ROOT / country_dir
        if not cpath.is_dir():
            continue
        for year_dir in sorted(p for p in cpath.iterdir() if p.is_dir()):
            year = year_dir.name
            kr_files = _find_files(year_dir / "KR", "KR")
            hr_files = _find_files(year_dir / "HR", "HR")
            gps_files = _find_files(year_dir / "GPS", "GPS")
            hr_by_recode = {}
            for hr in hr_files:
                parsed = parse_recode_stem(hr)
                if parsed:
                    hr_by_recode[parsed[2]] = hr
            gps_by_recode = {}
            for gps in gps_files:
                parsed = parse_recode_stem(gps)
                if parsed:
                    gps_by_recode[parsed[2]] = gps
            recodes = sorted({parse_recode_stem(p)[2] for p in kr_files if parse_recode_stem(p)})
            if not recodes and hr_files:
                recodes = sorted({parse_recode_stem(p)[2] for p in hr_files if parse_recode_stem(p)})
            if not recodes:
                recodes = ["MISSING"]
            for recode in recodes:
                kr = next((p for p in kr_files if parse_recode_stem(p) and parse_recode_stem(p)[2] == recode), None)
                hr = hr_by_recode.get(recode)
                gps = gps_by_recode.get(recode)
                if gps is None and len(gps_files) == 1:
                    gps = gps_files[0]
                rows.append(
                    {
                        "country": country,
                        "survey_year": year,
                        "recode": recode,
                        "kr_path": str(kr) if kr else "",
                        "hr_path": str(hr) if hr else "",
                        "gps_path": str(gps) if gps else "",
                        "has_kr": kr is not None,
                        "has_hr": hr is not None,
                        "has_gps": gps is not None,
                        "exclude_no_kr": (country, year) in EXCLUDE_NO_KR,
                    }
                )
    return pd.DataFrame(rows)


def _var_status(cols_lower: set[str], na_flags: dict[str, bool], names: list[str]) -> str:
    present = [n for n in names if n in cols_lower]
    if not present:
        return "absent"
    collected = [n for n in present if not na_flags.get(n, False)]
    if not collected:
        return "na_dictionary:" + ",".join(present)
    return "present:" + ",".join(collected)


def build_variable_dictionary(manifest: pd.DataFrame) -> pd.DataFrame:
    recs = []
    for row in manifest.itertuples(index=False):
        kr_cols: set[str] = set()
        hr_cols: set[str] = set()
        na_flags: dict[str, bool] = {}
        if row.kr_path:
            knames = list_dta_columns(Path(row.kr_path))
            kr_cols = set(map_lower(knames))
            na_flags.update(dta_labels_na(Path(row.kr_path)))
        if row.hr_path:
            hnames = list_dta_columns(Path(row.hr_path))
            hr_cols = set(map_lower(hnames))
            na_flags.update(dta_labels_na(Path(row.hr_path)))
        all_cols = kr_cols | hr_cols
        rec = {
            "country": row.country,
            "survey_year": row.survey_year,
            "recode": row.recode,
            "n_kr_cols": len(kr_cols),
            "n_hr_cols": len(hr_cols),
        }
        for key, names in INTEREST_VARS.items():
            rec[key] = _var_status(all_cols, na_flags, names)
        rdt_hr_slots = sum(1 for c in hr_cols if c.startswith("hml35_"))
        sh212_slots = sum(1 for c in hr_cols if c.startswith("sh212_"))
        rec["hml35_slots"] = rdt_hr_slots
        rec["sh212_slots"] = sh212_slots
        rec["rdt_source"] = (
            "hml35_hr"
            if rdt_hr_slots
            else ("sh212_hr" if sh212_slots else ("hml35_kr" if "hml35" in kr_cols else "none"))
        )
        recs.append(rec)
    return pd.DataFrame(recs)


def sample_interview_years(manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in manifest.itertuples(index=False):
        path = row.kr_path or row.hr_path
        if not path:
            continue
        p = Path(path)
        names = list_dta_columns(p)
        lower = map_lower(names)
        year_col = "v007" if "v007" in lower else ("hv007" if "hv007" in lower else None)
        if year_col is None:
            rows.append({**row._asdict(), "modal_year": pd.NA, "year_min": pd.NA, "year_max": pd.NA})
            continue
        df = read_dta(p, wanted=[year_col])
        years = pd.to_numeric(df[year_col], errors="coerce").dropna()
        # DHS sometimes stores 2-digit years
        years = years.where(years > 100, years + 1900)
        years = years.where(years > 1980, years + 100)
        if years.empty:
            modal = pd.NA
            ymin = ymax = pd.NA
        else:
            modal = int(years.mode().iloc[0])
            ymin, ymax = int(years.min()), int(years.max())
        d = row._asdict() if hasattr(row, "_asdict") else dict(row._asdict())
        d.update({"modal_year": modal, "year_min": ymin, "year_max": ymax})
        rows.append(d)
    return pd.DataFrame(rows)


def flag_duplicate_years(years: pd.DataFrame) -> pd.DataFrame:
    out = years.copy()
    out["keep"] = True
    out["dup_note"] = ""
    grouped = out.groupby(["country", "modal_year"], dropna=False)
    for (country, myear), g in grouped:
        if pd.isna(myear) or len(g) <= 1:
            continue
        # Prefer folder whose survey_year contains the modal year.
        scores = []
        for i, r in g.iterrows():
            folder = str(r["survey_year"])
            score = 0
            if str(int(myear)) in folder:
                score += 10
            if r["has_kr"] and r["has_hr"]:
                score += 3
            if r["has_gps"]:
                score += 1
            # Prefer later recode letters only as tie-break via recode string length.
            scores.append((score, i))
        scores.sort(reverse=True)
        winner = scores[0][1]
        for i, r in g.iterrows():
            if i == winner:
                continue
            out.loc[i, "keep"] = False
            out.loc[i, "dup_note"] = f"duplicate modal_year={int(myear)}; kept recode {out.loc[winner, 'recode']}"
    return out


def run_inventory() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ensure_dirs()
    manifest = discover_surveys()
    vardict = build_variable_dictionary(manifest)
    years = sample_interview_years(manifest)
    years = flag_duplicate_years(years)
    manifest.to_csv(TABLES / "survey_manifest.csv", index=False)
    vardict.to_csv(TABLES / "variable_dictionary.csv", index=False)
    years.to_csv(TABLES / "survey_years_dedup.csv", index=False)
    (TABLES / "inventory_summary.json").write_text(
        json.dumps(
            {
                "n_survey_rows": int(len(manifest)),
                "n_keep": int(years["keep"].sum()) if "keep" in years else int(len(years)),
                "n_drop_dup": int((~years["keep"]).sum()) if "keep" in years else 0,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Manifest: {len(manifest)} survey files")
    print(years[["country", "survey_year", "recode", "modal_year", "keep", "dup_note"]].to_string(index=False))
    return manifest, vardict, years


## Inlined `load_merge.py`


In [ ]:
"""Load KR+HR+GPS, reshape malaria person slots, flag merge failures (plan: 02_load_merge.py)."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd



def _lat_band(lat: pd.Series) -> pd.Series:
    x = pd.to_numeric(lat, errors="coerce")
    band = pd.Series(pd.NA, index=x.index, dtype="object")
    band = band.mask(x < 10, "guinean")
    band = band.mask((x >= 10) & (x < 14), "sudan")
    band = band.mask(x >= 14, "sahel")
    return band


def load_one(row: pd.Series) -> tuple[pd.DataFrame, dict]:
    stats = {
        "country": row.country,
        "survey_year": row.survey_year,
        "recode": row.recode,
        "n_kr": 0,
        "n_hr_hh": 0,
        "n_hr_person": 0,
        "n_gps": 0,
        "kr_no_line": 0,
        "kr_no_rdt_merge": 0,
        "rdt_no_kr": 0,
        "kr_no_gps": 0,
        "rdt_source": "none",
        "rdt_present": False,
    }
    if not row.has_kr or not row.has_hr or row.exclude_no_kr:
        stats["skip_reason"] = "no_kr_or_no_hr"
        return pd.DataFrame(), stats

    kr_path = Path(row.kr_path)
    hr_path = Path(row.hr_path)
    kr_avail = [c.lower() for c in list_dta_columns(kr_path)]
    hr_avail = [c.lower() for c in list_dta_columns(hr_path)]
    kr = read_dta(kr_path, wanted=wanted_kr_columns(kr_avail))
    hr = read_dta(hr_path, wanted=wanted_hr_columns(hr_avail))
    stats["n_kr"] = int(len(kr))
    stats["n_hr_hh"] = int(len(hr))

    hh_cols = [c for c in hr.columns if not any(c.startswith(p + "_") for p in ("hml35", "hml32", "hml12", "hml16", "sh212", "sh214r"))]
    hr_hh = hr[hh_cols].copy()
    person = reshape_person(hr)
    stats["n_hr_person"] = int(len(person))

    scheme = "standard"
    rdt_col = None
    if "hml35" in person.columns and person["hml35"].notna().any():
        rdt_col = "hml35"
        scheme = "standard"
    elif "sh212" in person.columns and person["sh212"].notna().any():
        rdt_col = "sh212"
        scheme = "sh212"
    stats["rdt_source"] = rdt_col or "none"
    stats["rdt_present"] = rdt_col is not None
    if rdt_col:
        person["rdt"] = recode_rdt(person[rdt_col], scheme)
        person["rdt_raw"] = pd.to_numeric(person[rdt_col], errors="coerce")
    else:
        person["rdt"] = pd.NA
        person["rdt_raw"] = pd.NA
    if "hml32" in person.columns:
        person["microscopy"] = recode_rdt(person["hml32"], "standard")
    else:
        person["microscopy"] = pd.NA
    if "hml12" in person.columns:
        person["hml12"] = pd.to_numeric(person["hml12"], errors="coerce")

    for col in ("v001", "v002", "b16", "hv001", "hv002"):
        if col in kr.columns:
            kr[col] = pd.to_numeric(kr[col], errors="coerce")
        if col in hr_hh.columns:
            hr_hh[col] = pd.to_numeric(hr_hh[col], errors="coerce")
        if col in person.columns:
            person[col] = pd.to_numeric(person[col], errors="coerce")

    stats["kr_no_line"] = int(kr["b16"].isna().sum()) if "b16" in kr.columns else int(len(kr))

    merged = kr.merge(
        hr_hh,
        left_on=["v001", "v002"],
        right_on=["hv001", "hv002"],
        how="left",
        suffixes=("", "_hr"),
    )
    if "hvidx" not in person.columns:
        person["hvidx"] = pd.NA
    merged = merged.merge(
        person,
        left_on=["v001", "v002", "b16"],
        right_on=["hv001", "hv002", "hvidx"],
        how="left",
        suffixes=("", "_p"),
    )
    stats["kr_no_rdt_merge"] = int(merged["rdt"].isna().sum()) if "rdt" in merged.columns else int(len(merged))

    tested = person.loc[person["rdt"].notna(), ["hv001", "hv002", "hvidx"]].copy()
    if len(tested) and "b16" in kr.columns:
        kr_keys = kr[["v001", "v002", "b16"]].dropna()
        t2 = tested.merge(
            kr_keys,
            left_on=["hv001", "hv002", "hvidx"],
            right_on=["v001", "v002", "b16"],
            how="left",
        )
        stats["rdt_no_kr"] = int(t2["b16"].isna().sum())
    else:
        stats["rdt_no_kr"] = int(len(tested))

    if row.gps_path:
        gps = read_gps_dbf(Path(row.gps_path))
        stats["n_gps"] = int(len(gps))
        if len(gps):
            merged = merged.merge(gps, on="hv001", how="left")
            stats["kr_no_gps"] = int(merged["gps_lat"].isna().sum()) if "gps_lat" in merged.columns else int(len(merged))
        else:
            merged["gps_lat"] = pd.NA
            merged["gps_lon"] = pd.NA
            stats["kr_no_gps"] = int(len(merged))
    else:
        merged["gps_lat"] = pd.NA
        merged["gps_lon"] = pd.NA
        stats["kr_no_gps"] = int(len(merged))

    merged["country"] = row.country
    merged["survey_year"] = row.survey_year
    merged["recode"] = row.recode
    merged["modal_year"] = row.modal_year if "modal_year" in row.index else pd.NA
    merged["antimalarial_any"] = antimalarial_any(merged)
    if "gps_lat" in merged.columns:
        merged["lat_band"] = _lat_band(merged["gps_lat"])
        merged["sahel_nigeria"] = np.where(
            (merged["country"] == "Nigeria") & (pd.to_numeric(merged["gps_lat"], errors="coerce") >= 11),
            1,
            np.where(merged["country"] == "Nigeria", 0, pd.NA),
        )
    else:
        merged["lat_band"] = pd.NA
        merged["sahel_nigeria"] = pd.NA
    return merged, stats


def run_merge() -> pd.DataFrame:
    ensure_dirs()
    _, _, years = run_inventory()
    keep = years[years["keep"]].copy()
    frames = []
    fail_rows = []
    for _, row in keep.iterrows():
        print(f"Loading {row.country} {row.survey_year} {row.recode} ...")
        df, stats = load_one(row)
        fail_rows.append(stats)
        if len(df):
            frames.append(df)
            print(f"  KR={stats['n_kr']:,} person={stats['n_hr_person']:,} rdt_source={stats['rdt_source']}")
        else:
            print(f"  skipped ({stats.get('skip_reason', '')})")
    fail = pd.DataFrame(fail_rows)
    fail.to_csv(TABLES / "merge_failures.csv", index=False)
    if not frames:
        raise RuntimeError("No surveys merged")
    pooled = pd.concat(frames, ignore_index=True, sort=False)
    out = DATA_DIR / "merged_raw.parquet"
    pooled.to_parquet(out, index=False)
    print(f"Pooled N={len(pooled):,} -> {out}")
    return pooled


## Inlined `study_population.py`


In [ ]:
"""Step 2 study population filters and Table 2 (plan: 03_study_population.py)."""

from __future__ import annotations

import json

import numpy as np
import pandas as pd



def _age_months(df: pd.DataFrame) -> pd.Series:
    if "b19" in df.columns:
        age = pd.to_numeric(df["b19"], errors="coerce")
    else:
        age = pd.Series(np.nan, index=df.index)
    if "hw1" in df.columns:
        hw1 = pd.to_numeric(df["hw1"], errors="coerce")
        age = age.fillna(hw1)
    if "hml16" in df.columns:
        age = age.fillna(pd.to_numeric(df["hml16"], errors="coerce"))
    return age


def apply_filters(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    work = df.copy()
    work["age_months"] = _age_months(work)
    work["weight"] = pd.to_numeric(work["v005"], errors="coerce") / 1_000_000 if "v005" in work.columns else np.nan
    if "hv005" in work.columns:
        alt = pd.to_numeric(work["hv005"], errors="coerce") / 1_000_000
        work["weight"] = work["weight"].fillna(alt) if hasattr(work["weight"], "fillna") else alt
    work["weight"] = pd.to_numeric(work["weight"], errors="coerce")
    work["h22"] = pd.to_numeric(work.get("h22"), errors="coerce") if "h22" in work.columns else np.nan
    work["rdt"] = pd.to_numeric(work.get("rdt"), errors="coerce")
    work["fever"] = np.where(work["h22"] == 1, 1, np.where(work["h22"] == 0, 0, np.nan))
    work["afebrile"] = work["fever"] == 0
    work["has_antimalarial_item"] = work["antimalarial_any"].notna() if "antimalarial_any" in work.columns else False

    flow_rows = []
    keys = ["country", "survey_year", "recode"]

    def add_step(name: str, mask: pd.Series) -> None:
        sub = work.loc[mask]
        g = (
            sub.groupby(keys, dropna=False)
            .agg(n=("age_months", "size"), rdt_pos=("rdt", lambda s: int((s == 1).sum())))
            .reset_index()
        )
        g["step"] = name
        flow_rows.append(g)

    m0 = pd.Series(True, index=work.index)
    add_step("0_kr_rows", m0)
    m1 = m0 & work["age_months"].between(6, 59)
    add_step("1_age_6_59", m1)
    m2 = m1 & work["rdt"].notna()
    add_step("2_valid_rdt", m2)
    m3 = m2 & work["h22"].notna()
    add_step("3_fever_item_nonmissing", m3)
    m4 = m3 & (work["fever"] == 0)
    add_step("4_afebrile_h22eq0", m4)
    # Antimalarial exclusion only where the item was asked (typically febrile children).
    # Among afebrile this is structurally missing; do not drop them for missing treatment.
    # Antimalarial items are asked after fever. Missing must not drop afebrile children
    # (pandas NA is not False: `~(NA==1)` is NA and would exclude the row).
    no_recent_am = ~work["antimalarial_any"].eq(1).fillna(False) if "antimalarial_any" in work.columns else True
    m5 = m4 & no_recent_am
    add_step("5_afebrile_no_recent_antimalarial", m5)

    flow = pd.concat(flow_rows, ignore_index=True)

    work["in_age"] = work["age_months"].between(6, 59)
    work["valid_rdt"] = work["rdt"].isin([0, 1])
    work["eval_sample"] = work["in_age"] & work["valid_rdt"] & work["fever"].isin([0, 1])
    work["train_sample"] = work["eval_sample"] & work["afebrile"] & no_recent_am

    t2_rows = []
    train_df = work.loc[work["train_sample"]]
    for (country, year, recode), g in train_df.groupby(keys, dropna=False):
        w = g["weight"].fillna(0) + 1e-12
        pos = int((g["rdt"] == 1).sum())
        t2_rows.append(
            {
                "country": country,
                "survey_year": year,
                "recode": recode,
                "n_afebrile": int(len(g)),
                "rdt_positive": pos,
                "rdt_negative": int((g["rdt"] == 0).sum()),
                "weighted_prev": float(np.average(g["rdt"], weights=w)),
                "unweighted_prev": float(g["rdt"].mean()),
                "flag_rdtpos_lt30": int(pos < RDT_POSITIVE_FLAG_N),
            }
        )
    table2 = pd.DataFrame(t2_rows)
    country_rows = []
    for country, g in train_df.groupby("country"):
        pos = int((g["rdt"] == 1).sum())
        country_rows.append(
            {
                "country": country,
                "n_afebrile": int(len(g)),
                "rdt_positive": pos,
                "flag_rdtpos_lt30": int(pos < RDT_POSITIVE_FLAG_N),
            }
        )
    country = pd.DataFrame(country_rows)
    return work, flow, table2.assign(_country_flag=table2["country"].map(dict(zip(country.country, country.flag_rdtpos_lt30))))


def write_phase1_report(work: pd.DataFrame, flow: pd.DataFrame, table2: pd.DataFrame, merge_fail: pd.DataFrame) -> None:
    lines = []
    lines.append("# Phase 1 dataset lock — Step 2 study population")
    lines.append("")
    lines.append("Date: 2026-08-22")
    lines.append("")
    lines.append("This report is the hard gate before modeling. The endpoint is malaria RDT positivity among children aged 6–59 months with no reported fever in the preceding two weeks (`h22==0`). Care-seeking items (`h32y`) are asked only after fever and are not independently informative in the afebrile subset.")
    lines.append("")
    lines.append("## Merge failures")
    lines.append("")
    if len(merge_fail):
        cols = [c for c in merge_fail.columns if c in {"country", "survey_year", "recode", "n_kr", "n_hr_person", "kr_no_rdt_merge", "rdt_no_kr", "kr_no_gps", "rdt_source", "rdt_present", "skip_reason"}]
        lines.append(df_to_md(merge_fail[cols]))
    lines.append("")
    lines.append("## Filter flow (N at each step)")
    lines.append("")
    pivot = flow.pivot_table(index=["country", "survey_year", "recode"], columns="step", values="n", aggfunc="sum")
    lines.append(df_to_md(pivot.reset_index()))
    lines.append("")
    lines.append("## Table 2 — afebrile subgroup with valid RDT")
    lines.append("")
    show = table2.drop(columns=["_country_flag"], errors="ignore")
    lines.append(df_to_md(show))
    lines.append("")
    flagged = table2[table2["flag_rdtpos_lt30"] == 1]
    lines.append("## Flags: country-year afebrile RDT+ < 30")
    lines.append("")
    if len(flagged):
        lines.append(df_to_md(flagged[["country", "survey_year", "recode", "n_afebrile", "rdt_positive"]]))
        lines.append("")
        lines.append("These survey-years are too small for stable calibration plots if analysed alone. They remain in the pooled training set unless a later decision excludes them from LOCO test folds.")
    else:
        lines.append("None.")
    country = (
        work.loc[work["train_sample"]]
        .groupby("country")
        .agg(n=("rdt", "size"), rdt_pos=("rdt", lambda s: int((s == 1).sum())))
        .reset_index()
    )
    lines.append("")
    lines.append("## Pooled by country (LOCO test-fold size)")
    lines.append("")
    country["flag_lt30"] = country["rdt_pos"] < 30
    lines.append(df_to_md(country))
    lines.append("")
    lines.append("## Notes")
    lines.append("")
    lines.append("- RDT lives in HR person slots (`hml35_nn`); KR has no `hml35`. Merge key: `v001+v002+b16` = `hv001+hv002+hvidx`.")
    lines.append("- Senegal 2008–09 uses `sh212` (1=positive, 2=negative) if `hml35` is absent.")
    lines.append("- Mali 2010 has no KR file and is excluded from the afebrile endpoint.")
    lines.append("- One-week fever recall is not collected; `h22` is a two-week window.")
    lines.append("- Recent antimalarial items are typically asked only of febrile children, so HRP2 persistence after treatment cannot be fully excluded in the afebrile group.")
    lines.append("- No rows were dropped silently; unmatched KR/RDT/GPS counts are in `outputs/tables/merge_failures.csv`.")
    REPORTS.joinpath("phase1_dataset_lock.md").write_text("\n".join(lines), encoding="utf-8")


def run_study_population(merged: pd.DataFrame | None = None) -> pd.DataFrame:
    ensure_dirs()
    if merged is None:
        path = DATA_DIR / "merged_raw.parquet"
        merged = pd.read_parquet(path) if path.exists() else run_merge()
    work, flow, table2 = apply_filters(merged)
    flow.to_csv(TABLES / "table2_flow.csv", index=False)
    table2.drop(columns=["_country_flag"], errors="ignore").to_csv(TABLES / "table2_afebrile.csv", index=False)
    fail_path = TABLES / "merge_failures.csv"
    merge_fail = pd.read_csv(fail_path) if fail_path.exists() else pd.DataFrame()
    write_phase1_report(work, flow, table2, merge_fail)
    out = DATA_DIR / "study_population.parquet"
    keep_cols = [c for c in work.columns if not c.endswith("_p") or c in work.columns[:5]]
    # store a slimmer frame for modeling
    slim_keep = [
        "country",
        "survey_year",
        "recode",
        "caseid",
        "v001",
        "v002",
        "b16",
        "v005",
        "weight",
        "v006",
        "v021",
        "v022",
        "v023",
        "v024",
        "v025",
        "v106",
        "v113",
        "v116",
        "v136",
        "v190",
        "b4",
        "age_months",
        "h22",
        "fever",
        "afebrile",
        "h32y",
        "h32z",
        "rdt",
        "microscopy",
        "hw53",
        "hml1",
        "hml12",
        "hv213",
        "hv214",
        "hv215",
        "hv253",
        "hv009",
        "gps_lat",
        "gps_lon",
        "lat_band",
        "sahel_nigeria",
        "antimalarial_any",
        "in_age",
        "valid_rdt",
        "eval_sample",
        "train_sample",
    ]
    slim = work[[c for c in slim_keep if c in work.columns]].copy()
    slim.to_parquet(out, index=False)
    print(f"Train (afebrile) N={int(work['train_sample'].sum()):,}")
    print(f"Eval (6-59 valid RDT, known fever) N={int(work['eval_sample'].sum()):,}")
    print(table2.drop(columns=["_country_flag"], errors="ignore").to_string(index=False))
    print(f"Wrote {REPORTS / 'phase1_dataset_lock.md'}")
    return slim


## Inlined `features.py`


In [ ]:
"""Predictor recodes, missingness, survey design IDs (plan: 04_features.py / 05)."""

from __future__ import annotations

import json

import numpy as np
import pandas as pd



def _housing_grp(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    out = pd.Series("missing", index=series.index, dtype="object")
    out = out.mask(x.between(10, 19), "natural")
    out = out.mask(x.between(20, 29), "rudimentary")
    out = out.mask(x.between(30, 39), "finished")
    out = out.mask(x.isna(), "missing")
    out = out.mask(~x.isna() & ~x.between(10, 39), "other")
    return out


def recode_features(df: pd.DataFrame) -> pd.DataFrame:
    out = nest_ids(df)
    out["wealth"] = pd.to_numeric(out.get("v190"), errors="coerce")
    out["urban"] = np.where(pd.to_numeric(out.get("v025"), errors="coerce") == 1, 1, np.where(pd.to_numeric(out.get("v025"), errors="coerce") == 2, 0, np.nan))
    out["educ"] = pd.to_numeric(out.get("v106"), errors="coerce")
    out["educ"] = out["educ"].where(out["educ"].between(0, 3), np.nan)
    hh = pd.to_numeric(out.get("v136"), errors="coerce")
    if "hv009" in out.columns:
        hh = hh.fillna(pd.to_numeric(out["hv009"], errors="coerce"))
    out["hhsize"] = hh
    water = pd.to_numeric(out.get("v113"), errors="coerce")
    toilet = pd.to_numeric(out.get("v116"), errors="coerce")
    out["water_improved"] = np.where(water.isna() | water.isin([96, 99]), np.nan, water.isin(IMPROVED_WATER).astype(float))
    out["toilet_improved"] = np.where(toilet.isna() | toilet.isin([96, 99]), np.nan, toilet.isin(IMPROVED_TOILET).astype(float))
    nets = pd.to_numeric(out.get("hml1"), errors="coerce")
    out["net_own"] = np.where(nets.isna(), np.nan, (nets > 0).astype(float))
    hml12 = pd.to_numeric(out.get("hml12"), errors="coerce")
    out["net_used"] = np.where(hml12.isna(), np.nan, hml12.isin(NET_USED_CODES).astype(float))
    out["floor_grp"] = _housing_grp(out["hv213"]) if "hv213" in out.columns else pd.Series("missing", index=out.index)
    out["wall_grp"] = _housing_grp(out["hv214"]) if "hv214" in out.columns else pd.Series("missing", index=out.index)
    out["roof_grp"] = _housing_grp(out["hv215"]) if "hv215" in out.columns else pd.Series("missing", index=out.index)
    out["female"] = np.where(pd.to_numeric(out.get("b4"), errors="coerce") == 2, 1, np.where(pd.to_numeric(out.get("b4"), errors="coerce") == 1, 0, np.nan))
    month = pd.to_numeric(out.get("v006"), errors="coerce")
    if "hv006" in out.columns:
        month = month.fillna(pd.to_numeric(out["hv006"], errors="coerce"))
    out["month"] = month
    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)
    out["lat_band"] = out["lat_band"].fillna("missing").astype(str)
    ng = pd.to_numeric(out.get("sahel_nigeria"), errors="coerce")
    out["sahel_nigeria"] = np.where(out["country"] != "Nigeria", 0.0, ng)
    if "hw53" in out.columns:
        hb = pd.to_numeric(out["hw53"], errors="coerce")
        hb = hb.where(~hb.isin([999, 9999, 99]), np.nan)
        # Stored with one implied decimal when values look like 30–200.
        hb = np.where(hb > 25, hb / 10.0, hb)
        out[HB_FEATURE] = hb
    else:
        out[HB_FEATURE] = np.nan
    if "hv253" in out.columns:
        irs = pd.to_numeric(out["hv253"], errors="coerce")
        out["irs"] = np.where(irs.isin([0, 1]), irs, np.nan)
    else:
        out["irs"] = np.nan
    return out


def missingness_table(df: pd.DataFrame) -> pd.DataFrame:
    keys = ["country", "survey_year"]
    cols = [c for c in PRIMARY_FEATURES + [HB_FEATURE, "irs"] if c in df.columns]
    rows = []
    for (country, year), g in df.groupby(keys, dropna=False):
        rec = {"country": country, "survey_year": year, "n": int(len(g))}
        for c in cols:
            rec[f"{c}_miss_pct"] = float(g[c].isna().mean() * 100) if c in g.columns else 100.0
        rec["primary_mean_miss_pct"] = float(np.mean([rec[f"{c}_miss_pct"] for c in PRIMARY_FEATURES if f"{c}_miss_pct" in rec]))
        rec["drop_gt20pct_predictors"] = int(rec["primary_mean_miss_pct"] > 20)
        rows.append(rec)
    return pd.DataFrame(rows)


def run_features(pop: pd.DataFrame | None = None) -> pd.DataFrame:
    ensure_dirs()
    if pop is None:
        pop = pd.read_parquet(DATA_DIR / "study_population.parquet")
    feat = recode_features(pop)
    miss = missingness_table(feat.loc[feat["train_sample"]])
    miss.to_csv(TABLES / "missingness_by_survey.csv", index=False)
    feat.to_parquet(DATA_DIR / "analytic.parquet", index=False)
    (TABLES / "missingness_summary.json").write_text(
        json.dumps(
            {
                "n_train": int(feat["train_sample"].sum()),
                "n_eval": int(feat["eval_sample"].sum()),
                "surveys_gt20pct": miss.loc[miss["drop_gt20pct_predictors"] == 1, ["country", "survey_year"]].to_dict("records"),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(miss[["country", "survey_year", "n", "primary_mean_miss_pct", "drop_gt20pct_predictors"]].to_string(index=False))
    return feat


## Inlined `models_loco.py`


In [ ]:
"""Leave-one-country-out models, survey-weighted metrics, incremental DCA (plan: 06–07)."""

from __future__ import annotations

import json
from dataclasses import dataclass

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


NUM_COLS = [
    "wealth",
    "urban",
    "educ",
    "hhsize",
    "water_improved",
    "toilet_improved",
    "net_own",
    "net_used",
    "age_months",
    "female",
    "month_sin",
    "month_cos",
    "sahel_nigeria",
]
CAT_COLS = ["floor_grp", "wall_grp", "roof_grp", "lat_band"]


def equal_country_weights(df: pd.DataFrame) -> pd.DataFrame:
    """Rescale native DHS weights so each country has the same total weight mass.

    Evaluation metrics on a single held-out country are invariant to a constant
    scale, so this is applied only to training folds (see ``run_loco``).
    """
    out = df.copy()
    w = pd.to_numeric(out["weight"], errors="coerce").fillna(0).clip(lower=0.0)
    tot = w.groupby(out["country"]).transform("sum")
    n_c = max(int(out["country"].nunique()), 1)
    grand = float(w.sum())
    scale = np.where(tot > 0, (grand / n_c) / tot, 0.0)
    out["weight"] = w.to_numpy(dtype=float) * scale
    return out


def _xy(df: pd.DataFrame, extra_num: list[str] | None = None) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    cols = [c for c in NUM_COLS + CAT_COLS if c in df.columns]
    if extra_num:
        cols = cols + [c for c in extra_num if c in df.columns and c not in cols]
    X = df[cols].copy()
    for c in CAT_COLS:
        if c in X.columns:
            X[c] = X[c].astype(str).fillna("missing")
    y = pd.to_numeric(df["rdt"], errors="coerce").to_numpy()
    w = pd.to_numeric(df["weight"], errors="coerce").fillna(0).to_numpy()
    w = np.where(w > 0, w, 0.0)
    return X, y, w


def make_preprocessor(num_cols: list[str], cat_cols: list[str], iterative: bool = True) -> ColumnTransformer:
    if iterative:
        num_imp = IterativeImputer(random_state=SEED, max_iter=5, skip_complete=True)
    else:
        num_imp = SimpleImputer(strategy="median")
    num = Pipeline([("impute", num_imp), ("scale", StandardScaler())])
    cat = Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [("num", num, num_cols), ("cat", cat, cat_cols)],
        remainder="drop",
    )


def weighted_auc(y, p, w) -> float:
    y = np.asarray(y)
    p = np.asarray(p)
    w = np.asarray(w)
    mask = np.isfinite(y) & np.isfinite(p) & np.isfinite(w) & (w >= 0)
    if mask.sum() < 10 or len(np.unique(y[mask])) < 2:
        return float("nan")
    try:
        return float(roc_auc_score(y[mask], p[mask], sample_weight=w[mask]))
    except ValueError:
        return float("nan")


def oe_ratio(y, p, w) -> float:
    w = np.asarray(w, dtype=float)
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p) & (w > 0)
    if mask.sum() == 0:
        return float("nan")
    exp = np.average(p[mask], weights=w[mask])
    obs = np.average(y[mask], weights=w[mask])
    if exp <= 0:
        return float("nan")
    return float(obs / exp)


def calib_slope_intercept(y, p, w) -> tuple[float, float]:
    y = np.asarray(y)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p) & (w > 0)
    if mask.sum() < 20 or len(np.unique(y[mask])) < 2:
        return float("nan"), float("nan")
    lp = np.log(p[mask] / (1 - p[mask])).reshape(-1, 1)
    try:
        lr = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=500)
        lr.fit(lp, y[mask], sample_weight=w[mask])
        return float(lr.coef_[0, 0]), float(lr.intercept_[0])
    except Exception:
        return float("nan"), float("nan")


def calibration_bins(y, p, w, n_bins: int = CALIBRATION_BINS) -> pd.DataFrame:
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p) & (w > 0)
    y, p, w = y[mask], p[mask], w[mask]
    if len(y) < n_bins:
        return pd.DataFrame()
    qs = np.quantile(p, np.linspace(0, 1, n_bins + 1))
    qs[0], qs[-1] = 0, 1
    rows = []
    for i in range(n_bins):
        m = (p >= qs[i]) & (p <= qs[i + 1] if i == n_bins - 1 else p < qs[i + 1])
        if m.sum() == 0:
            continue
        rows.append(
            {
                "bin": i + 1,
                "n": int(m.sum()),
                "pred": float(np.average(p[m], weights=w[m])),
                "obs": float(np.average(y[m], weights=w[m])),
            }
        )
    return pd.DataFrame(rows)


def net_benefit(y, tested, w, t: float) -> float:
    y = np.asarray(y, dtype=float)
    tested = np.asarray(tested, dtype=bool)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(y) & np.isfinite(w) & (w > 0)
    y, tested, w = y[mask], tested[mask], w[mask]
    if w.sum() <= 0:
        return float("nan")
    tp = np.average((y == 1) & tested, weights=w)
    fp = np.average((y == 0) & tested, weights=w)
    return float(tp - fp * (t / (1 - t)))


def dca_curves(y, fever, p_afebrile, w, afebrile_mask) -> pd.DataFrame:
    rows = []
    fever = np.asarray(fever) == 1
    p = np.asarray(p_afebrile, dtype=float)
    afe = np.asarray(afebrile_mask, dtype=bool)
    for t in DCA_THRESHOLDS:
        combo = fever | (afe & np.isfinite(p) & (p >= t))
        rows.append(
            {
                "threshold": t,
                "nb_none": 0.0,
                "nb_all": net_benefit(y, np.ones(len(y), dtype=bool), w, t),
                "nb_fever": net_benefit(y, fever, w, t),
                "nb_fever_plus_model": net_benefit(y, combo, w, t),
            }
        )
    return pd.DataFrame(rows)


def cluster_bootstrap_ci(df: pd.DataFrame, pcol: str, n_boot: int = N_BOOTSTRAP) -> dict:
    rng = np.random.default_rng(SEED)
    y = pd.to_numeric(df["rdt"], errors="coerce").to_numpy()
    p = pd.to_numeric(df[pcol], errors="coerce").to_numpy()
    w = pd.to_numeric(df["weight"], errors="coerce").to_numpy()
    psu = pd.factorize(df["psu"].astype(str))[0]
    n_psu = int(psu.max()) + 1 if len(psu) else 0
    if n_psu < 5 or n_boot < 1:
        return {"auc_ci_low": float("nan"), "auc_ci_high": float("nan"), "oe_ci_low": float("nan"), "oe_ci_high": float("nan")}
    groups = [np.flatnonzero(psu == i) for i in range(n_psu)]
    aucs, oes = [], []
    for _ in range(n_boot):
        draw = rng.integers(0, n_psu, size=n_psu)
        idx = np.concatenate([groups[i] for i in draw], axis=0)
        aucs.append(weighted_auc(y[idx], p[idx], w[idx]))
        oes.append(oe_ratio(y[idx], p[idx], w[idx]))

    def ci(vals):
        vals = np.array([v for v in vals if np.isfinite(v)])
        if len(vals) < 20:
            return float("nan"), float("nan")
        return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

    a_lo, a_hi = ci(aucs)
    o_lo, o_hi = ci(oes)
    return {"auc_ci_low": a_lo, "auc_ci_high": a_hi, "oe_ci_low": o_lo, "oe_ci_high": o_hi}


def _fit_enet(X, y, w, num_cols, cat_cols, params) -> Pipeline:
    pipe = Pipeline(
        [
            ("prep", make_preprocessor(num_cols, cat_cols, iterative=False)),
            (
                "clf",
                    LogisticRegression(
                        solver="saga",
                        l1_ratio=params["l1_ratio"],
                        C=params["C"],
                        max_iter=800,
                        random_state=SEED,
                    ),
            ),
        ]
    )
    pipe.fit(X, y, clf__sample_weight=w)
    return pipe


def _fit_lgb(X, y, w, num_cols, cat_cols, params) -> Pipeline:
    pipe = Pipeline(
        [
            ("prep", make_preprocessor(num_cols, cat_cols, iterative=False)),
            (
                "clf",
                lgb.LGBMClassifier(
                    objective="binary",
                    random_state=SEED,
                    verbosity=-1,
                    **params,
                ),
            ),
        ]
    )
    pipe.fit(X, y, clf__sample_weight=w)
    return pipe


@dataclass
class FoldResult:
    country: str
    model: str
    params: dict
    auc: float
    oe: float
    slope: float
    intercept: float


def inner_select(train: pd.DataFrame, extra_num: list[str] | None = None, equal_country_weight: bool = False):
    countries = sorted(train["country"].unique())
    num_cols = [c for c in NUM_COLS if c in train.columns]
    if extra_num:
        num_cols = num_cols + [c for c in extra_num if c in train.columns]
    cat_cols = [c for c in CAT_COLS if c in train.columns]
    best_enet, best_enet_s = {"l1_ratio": 0.5, "C": 1.0}, float("nan")
    best_lgb, best_lgb_s = LGBM_GRID[0], -1
    for params in LGBM_GRID:
        scores = []
        for val_c in countries:
            tr = train[train["country"] != val_c]
            if equal_country_weight:
                tr = equal_country_weights(tr)
            va = train[train["country"] == val_c]
            if va["rdt"].nunique() < 2 or (va["rdt"] == 1).sum() < 10:
                continue
            Xtr, ytr, wtr = _xy(tr, extra_num)
            Xva, yva, wva = _xy(va, extra_num)
            try:
                m = _fit_lgb(Xtr, ytr, wtr, num_cols, cat_cols, params)
                p = m.predict_proba(Xva)[:, 1]
                scores.append(weighted_auc(yva, p, wva))
            except Exception:
                continue
        s = float(np.nanmean(scores)) if scores else float("nan")
        if np.isfinite(s) and s > best_lgb_s:
            best_lgb, best_lgb_s = params, s
    if best_enet is None:
        best_enet = {"l1_ratio": 0.5, "C": 1.0}
    return best_enet, best_lgb, num_cols, cat_cols


def _predict_fold(train, test, extra_num=None, equal_country_weight: bool = False):
    best_enet, best_lgb, num_cols, cat_cols = inner_select(
        train, extra_num, equal_country_weight=equal_country_weight
    )
    Xtr, ytr, wtr = _xy(train, extra_num)
    Xte, yte, wte = _xy(test, extra_num)
    enet = _fit_enet(Xtr, ytr, wtr, num_cols, cat_cols, best_enet)
    gbm = _fit_lgb(Xtr, ytr, wtr, num_cols, cat_cols, best_lgb)
    p_enet = enet.predict_proba(Xte)[:, 1]
    p_lgb = gbm.predict_proba(Xte)[:, 1]
    return p_enet, p_lgb, enet, gbm, best_enet, best_lgb


def _metrics_block(df, pcol, model, country, extra=None, n_boot: int = N_BOOTSTRAP) -> dict:
    y, p, w = df["rdt"].to_numpy(), df[pcol].to_numpy(), df["weight"].to_numpy()
    slope, intercept = calib_slope_intercept(y, p, w)
    rec = {
        "country": country,
        "model": model,
        "n": int(len(df)),
        "rdt_pos": int((df["rdt"] == 1).sum()),
        "auc": weighted_auc(y, p, w),
        "oe": oe_ratio(y, p, w),
        "slope": slope,
        "intercept": intercept,
        "flag_pos_lt50": int((df["rdt"] == 1).sum() < SUBGROUP_RDT_POS_FLAG_N),
    }
    if extra:
        rec.update(extra)
    rec.update(cluster_bootstrap_ci(df, pcol, n_boot=n_boot))
    return rec


def _params_from_primary(country: str) -> tuple[dict, dict]:
    path = TABLES / "metrics_loco_primary.csv"
    if not path.exists():
        return {"l1_ratio": 0.5, "C": 1.0}, LGBM_GRID[1]
    m = pd.read_csv(path)
    en = m[(m["country"] == country) & (m["model"] == "elasticnet")]
    lg = m[(m["country"] == country) & (m["model"] == "lightgbm")]
    pe = json.loads(en.iloc[0]["params"]) if len(en) and pd.notna(en.iloc[0].get("params")) else {"l1_ratio": 0.5, "C": 1.0}
    pl = json.loads(lg.iloc[0]["params"]) if len(lg) and pd.notna(lg.iloc[0].get("params")) else LGBM_GRID[1]
    return pe, pl


def run_loco(
    analytic: pd.DataFrame | None = None,
    extra_num: list[str] | None = None,
    tag: str = "primary",
    tune: bool = True,
    equal_country_weight: bool = False,
    overwrite: bool = False,
    reuse_primary_params: bool = False,
    n_boot: int | None = None,
    compute_aux: bool = True,
) -> dict:
    ensure_dirs()
    dest = TABLES / f"metrics_loco_{tag}.csv"
    if tag == "primary" and dest.exists() and not overwrite:
        print(f"Primary LOCO metrics already exist at {dest}; skipping to protect Table 2. Pass overwrite=True to refit.")
        preds_path = DATA_DIR / "predictions_primary.parquet"
        return {
            "preds": pd.read_parquet(preds_path) if preds_path.exists() else pd.DataFrame(),
            "metrics": pd.read_csv(dest),
            "calib": pd.read_csv(TABLES / "calibration_bins_primary.csv") if (TABLES / "calibration_bins_primary.csv").exists() else pd.DataFrame(),
            "dca": pd.read_csv(TABLES / "dca_primary.csv") if (TABLES / "dca_primary.csv").exists() else pd.DataFrame(),
            "sub": pd.read_csv(TABLES / "subgroup_metrics_primary.csv") if (TABLES / "subgroup_metrics_primary.csv").exists() else pd.DataFrame(),
            "gbm": None,
        }
    if n_boot is None:
        n_boot = N_BOOTSTRAP
    if analytic is None:
        analytic = pd.read_parquet(DATA_DIR / "analytic.parquet")
    train_all = analytic.loc[analytic["train_sample"]].copy()
    eval_all = analytic.loc[analytic["eval_sample"]].copy()
    countries = sorted(train_all["country"].unique())
    pred_frames = []
    metric_rows = []
    calib_rows = []
    dca_rows = []
    subgroup_rows = []
    last_gbm = None
    last_cols = None
    for held in countries:
        print(f"[{tag}] LOCO hold-out {held}")
        tr = train_all[train_all["country"] != held]
        if equal_country_weight:
            tr = equal_country_weights(tr)
        te_trainish = train_all[train_all["country"] == held]
        te_eval = eval_all[eval_all["country"] == held]
        if (te_trainish["rdt"] == 1).sum() < 10:
            print(f"  skip {held}: too few afebrile RDT+")
            continue
        if reuse_primary_params:
            pe, pl = _params_from_primary(held)
            num_cols = [c for c in NUM_COLS if c in tr.columns]
            if extra_num:
                num_cols = num_cols + [c for c in extra_num if c in tr.columns]
            cat_cols = [c for c in CAT_COLS if c in tr.columns]
            Xtr, ytr, wtr = _xy(tr, extra_num)
            Xte, yte, wte = _xy(te_trainish, extra_num)
            enet = _fit_enet(Xtr, ytr, wtr, num_cols, cat_cols, pe)
            gbm = _fit_lgb(Xtr, ytr, wtr, num_cols, cat_cols, pl)
            p_enet = enet.predict_proba(Xte)[:, 1]
            p_lgb = gbm.predict_proba(Xte)[:, 1]
        elif tune:
            p_enet, p_lgb, enet, gbm, pe, pl = _predict_fold(
                tr, te_trainish, extra_num, equal_country_weight=equal_country_weight
            )
        else:
            num_cols = [c for c in NUM_COLS if c in tr.columns]
            if extra_num:
                num_cols = num_cols + [c for c in extra_num if c in tr.columns]
            cat_cols = [c for c in CAT_COLS if c in tr.columns]
            pe = {"l1_ratio": 0.5, "C": 1.0}
            pl = LGBM_GRID[1]
            Xtr, ytr, wtr = _xy(tr, extra_num)
            Xte, yte, wte = _xy(te_trainish, extra_num)
            enet = _fit_enet(Xtr, ytr, wtr, num_cols, cat_cols, pe)
            gbm = _fit_lgb(Xtr, ytr, wtr, num_cols, cat_cols, pl)
            p_enet = enet.predict_proba(Xte)[:, 1]
            p_lgb = gbm.predict_proba(Xte)[:, 1]
        last_gbm, last_cols = gbm, extra_num
        te = te_trainish.copy()
        te["p_enet"] = p_enet
        te["p_lgb"] = p_lgb
        ev = te_eval.copy()
        if compute_aux:
            afe = ev["fever"].eq(0)
            if afe.any():
                Xafe, _, _ = _xy(ev.loc[afe], extra_num)
                ev.loc[afe, "p_enet"] = enet.predict_proba(Xafe)[:, 1]
                ev.loc[afe, "p_lgb"] = gbm.predict_proba(Xafe)[:, 1]
        pred_frames.append(te)
        for model, pcol in [("elasticnet", "p_enet"), ("lightgbm", "p_lgb")]:
            metric_rows.append(_metrics_block(te, pcol, model, held, extra={"params": json.dumps(pe if model == "elasticnet" else pl), "tag": tag}, n_boot=n_boot))
            if not compute_aux:
                continue
            bins = calibration_bins(te["rdt"], te[pcol], te["weight"])
            bins["country"] = held
            bins["model"] = model
            bins["tag"] = tag
            calib_rows.append(bins)
            dca = dca_curves(
                ev["rdt"].to_numpy(),
                ev["fever"].to_numpy(),
                ev[pcol].to_numpy() if pcol in ev.columns else np.full(len(ev), np.nan),
                ev["weight"].to_numpy(),
                ev["afebrile"].to_numpy() if "afebrile" in ev.columns else (ev["fever"] == 0).to_numpy(),
            )
            dca["country"] = held
            dca["model"] = model
            dca["tag"] = tag
            dca_rows.append(dca)
            for wealth, g in te.groupby("wealth"):
                if len(g) < 30:
                    continue
                rec = _metrics_block(g, pcol, model, held, extra={"subgroup": f"wealth_{int(wealth) if pd.notna(wealth) else 'na'}", "tag": tag}, n_boot=200)
                subgroup_rows.append(rec)
            for urb, g in te.groupby("urban"):
                if len(g) < 30:
                    continue
                rec = _metrics_block(g, pcol, model, held, extra={"subgroup": f"urban_{int(urb) if pd.notna(urb) else 'na'}", "tag": tag}, n_boot=200)
                subgroup_rows.append(rec)

    preds = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    metrics = pd.DataFrame(metric_rows)
    calib = pd.concat(calib_rows, ignore_index=True) if calib_rows else pd.DataFrame()
    dca = pd.concat(dca_rows, ignore_index=True) if dca_rows else pd.DataFrame()
    sub = pd.DataFrame(subgroup_rows)
    preds.to_parquet(DATA_DIR / f"predictions_{tag}.parquet", index=False)
    metrics.to_csv(TABLES / f"metrics_loco_{tag}.csv", index=False)
    calib.to_csv(TABLES / f"calibration_bins_{tag}.csv", index=False)
    dca.to_csv(TABLES / f"dca_{tag}.csv", index=False)
    sub.to_csv(TABLES / f"subgroup_metrics_{tag}.csv", index=False)
    if last_gbm is not None:
        import joblib

        joblib.dump(last_gbm, MODELS_DIR / f"lgb_lastfold_{tag}.joblib")
    print(metrics.to_string(index=False))
    return {"preds": preds, "metrics": metrics, "calib": calib, "dca": dca, "sub": sub, "gbm": last_gbm}


def run_senegal_temporal(analytic: pd.DataFrame | None = None) -> pd.DataFrame:
    if analytic is None:
        analytic = pd.read_parquet(DATA_DIR / "analytic.parquet")
    sn = analytic[(analytic["country"] == "Senegal") & analytic["train_sample"]].copy()
    test_mask = sn["survey_year"].astype(str).str.contains("2020")
    tr = sn[~test_mask]
    te = sn[test_mask]
    if len(tr) < 100 or (te["rdt"] == 1).sum() < 5:
        print("Senegal temporal: insufficient sample; writing empty table")
        out = pd.DataFrame()
        out.to_csv(TABLES / "senegal_temporal.csv", index=False)
        return out
    print(f"Senegal temporal train N={len(tr)} test N={len(te)} RDT+={int((te.rdt==1).sum())}")
    p_enet, p_lgb, _, _, pe, pl = _predict_fold(tr, te)
    te = te.copy()
    te["p_enet"] = p_enet
    te["p_lgb"] = p_lgb
    rows = []
    for model, pcol in [("elasticnet", "p_enet"), ("lightgbm", "p_lgb")]:
        rows.append(_metrics_block(te, pcol, model, "Senegal-temporal"))
    out = pd.DataFrame(rows)
    out.to_csv(TABLES / "senegal_temporal.csv", index=False)
    te.to_parquet(DATA_DIR / "predictions_senegal_temporal.parquet", index=False)
    return out


## Inlined `shap_explain.py`


In [ ]:
"""SHAP for the gradient boosting model (plan: 08). Model behavior, not causal biology."""

from __future__ import annotations

import numpy as np
import pandas as pd
import shap
from sklearn.pipeline import Pipeline


SHAP_LABELS = {
    "cat__lat_band_sahel": "Latitude band (Sahel)",
    "num__month_sin": "Survey month (sine)",
    "num__month_cos": "Survey month (cosine)",
    "num__urban": "Urban residence",
    "num__hhsize": "Household size",
    "num__age_months": "Child age (months)",
    "num__toilet_improved": "Improved sanitation",
    "num__educ": "Maternal education (years)",
    "num__wealth": "Wealth quintile",
    "cat__floor_grp_finished": "Floor: finished",
    "cat__floor_grp_natural": "Floor: natural",
    "cat__wall_grp_finished": "Wall: finished",
    "cat__roof_grp_finished": "Roof: finished",
    "cat__lat_band_guinean": "Latitude band (Guinean)",
    "cat__wall_grp_natural": "Wall: natural",
    "num__net_used": "Bed-net used last night",
    "num__net_own": "Bed-net ownership",
    "num__female": "Female child",
    "num__water_improved": "Improved water",
    "num__sahel_nigeria": "Nigeria Sahel band",
}


def _shap_label(name: str) -> str:
    if name in SHAP_LABELS:
        return SHAP_LABELS[name]
    cleaned = name.replace("cat__", "").replace("num__", "").replace("_", " ")
    return cleaned


def run_shap(analytic: pd.DataFrame | None = None, sample_n: int = 4000) -> None:
    ensure_dirs()
    if analytic is None:
        analytic = pd.read_parquet(DATA_DIR / "analytic.parquet")
    train = analytic.loc[analytic["train_sample"]].copy()
    best_lgb = LGBM_GRID[1]
    num_cols = [c for c in NUM_COLS if c in train.columns]
    cat_cols = [c for c in CAT_COLS if c in train.columns]
    X, y, w = _xy(train)
    gbm: Pipeline = _fit_lgb(X, y, w, num_cols, cat_cols, best_lgb)
    Xt = gbm.named_steps["prep"].transform(X)
    feature_names = gbm.named_steps["prep"].get_feature_names_out()
    clf = gbm.named_steps["clf"]
    rng = np.random.default_rng(SEED)
    n = min(sample_n, Xt.shape[0])
    idx = rng.choice(Xt.shape[0], size=n, replace=False)
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(Xt[idx])
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    mean_abs = np.abs(shap_values).mean(axis=0)
    importance = (
        pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )
    importance.to_csv(TABLES / "shap_importance.csv", index=False)
    import matplotlib.pyplot as plt

    labels = [_shap_label(f) for f in feature_names]
    top = importance.head(15).copy()
    top["label"] = top["feature"].map(_shap_label)

    plt.rcParams.update(
        {
            "font.size": 9,
            "axes.labelsize": 9,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
            "axes.linewidth": 1.2,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    fig, ax = plt.subplots(figsize=(90 / 25.4 * 2 + 0.2, 110 / 25.4))
    ax.barh(
        top["label"][::-1],
        top["mean_abs_shap"][::-1],
        color="#08306B",
        edgecolor="#111111",
        linewidth=0.4,
        height=0.72,
    )
    ax.set_xlabel("Mean |SHAP|")
    ax.text(0.0, 1.02, "A", transform=ax.transAxes, fontsize=11, fontweight="bold", va="bottom")
    fig.tight_layout()
    fig.savefig(FIGURES / "figure5_shap_bar.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    plt.figure(figsize=(90 / 25.4 * 2 + 0.2, 110 / 25.4))
    shap.summary_plot(shap_values, Xt[idx], feature_names=labels, max_display=15, show=False)
    fig = plt.gcf()
    fig.text(0.01, 0.98, "B", fontsize=11, fontweight="bold", va="top")
    plt.tight_layout()
    plt.savefig(FIGURES / "figure5_shap_beeswarm.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(importance.head(15).to_string(index=False))
    print("SHAP describes how the fitted model uses features; it is not a causal statement about malaria biology.")


## Inlined `sensitivity.py`


In [ ]:
"""Sensitivity analyses (plan: 09). Not headline results."""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split



def microscopy_sensitivity(analytic: pd.DataFrame) -> pd.DataFrame:
    mask = pd.Series(False, index=analytic.index)
    for country, year in MICROSCOPY_HEADLINE:
        mask |= (analytic["country"] == country) & (analytic["survey_year"].astype(str) == str(year))
    sub = analytic.loc[mask].copy()
    sub = sub[sub["microscopy"].isin([0, 1])]
    sub["rdt"] = sub["microscopy"]
    sub["train_sample"] = sub["train_sample"] & sub["microscopy"].isin([0, 1])
    sub["eval_sample"] = sub["eval_sample"] & sub["microscopy"].isin([0, 1])
    print(f"Microscopy sensitivity N train={int(sub.train_sample.sum())} countries={list(sub.country.unique())}")
    if sub.loc[sub.train_sample, "country"].nunique() < 2:
        out = pd.DataFrame({"note": ["Need two countries with microscopy for LOCO"]})
        out.to_csv(TABLES / "sensitivity_microscopy.csv", index=False)
        return out
    res = run_loco(sub, tag="microscopy", tune=False)
    res["metrics"].to_csv(TABLES / "sensitivity_microscopy.csv", index=False)
    return res["metrics"]


def hb_sensitivity(analytic: pd.DataFrame) -> pd.DataFrame:
    sub = analytic.copy()
    print("Hb-in-model sensitivity (not primary)")
    res = run_loco(sub, extra_num=["hb_gdl"], tag="hb", tune=False)
    res["metrics"].to_csv(TABLES / "sensitivity_hb.csv", index=False)
    return res["metrics"]


def drop_incomplete_surveys(analytic: pd.DataFrame) -> pd.DataFrame:
    miss = pd.read_csv(TABLES / "missingness_by_survey.csv")
    drop = miss.loc[miss["drop_gt20pct_predictors"] == 1, ["country", "survey_year"]]
    drop.to_csv(TABLES / "sensitivity_dropped_surveys.csv", index=False)
    if drop.empty:
        print("No surveys with >20% mean primary-predictor missingness")
        pd.DataFrame({"note": ["none dropped"]}).to_csv(TABLES / "sensitivity_incomplete.csv", index=False)
        return pd.DataFrame()
    key = set(zip(drop.country, drop.survey_year.astype(str)))
    keep = analytic[~analytic.apply(lambda r: (r.country, str(r.survey_year)) in key, axis=1)].copy()
    res = run_loco(keep, tag="incomplete_drop", tune=False)
    res["metrics"].to_csv(TABLES / "sensitivity_incomplete.csv", index=False)
    return res["metrics"]


def fever_dontknow_sensitivity(analytic: pd.DataFrame) -> pd.DataFrame:
    """Primary excludes h22==8. Alternate: treat don't-know as afebrile."""
    alt = analytic.copy()
    h22 = pd.to_numeric(alt.get("h22"), errors="coerce")
    extra = alt["in_age"] & alt["valid_rdt"] & h22.eq(8)
    n_extra = int(extra.sum())
    alt.loc[extra, "afebrile"] = True
    alt.loc[extra, "fever"] = 0
    alt.loc[extra, "train_sample"] = extra
    alt.loc[extra, "eval_sample"] = extra | alt["eval_sample"]
    print(f"Alternate fever coding: added {n_extra} children with h22==8 as afebrile")
    pd.DataFrame(
        {
            "definition": ["primary_h22eq0", "alternate_h22in_0_or_8"],
            "n_train": [int(analytic["train_sample"].sum()), int(alt["train_sample"].sum())],
            "rdt_pos_train": [
                int((analytic.loc[analytic.train_sample, "rdt"] == 1).sum()),
                int((alt.loc[alt.train_sample, "rdt"] == 1).sum()),
            ],
            "n_added_dontknow": [0, n_extra],
        }
    ).to_csv(TABLES / "sensitivity_fever_window.csv", index=False)
    # One-week recall does not exist in these files; this is the pre-specified substitute.
    if n_extra >= 50:
        res = run_loco(alt, tag="fever_dk", tune=False)
        res["metrics"].to_csv(TABLES / "sensitivity_fever_dk_metrics.csv", index=False)
        return res["metrics"]
    return pd.DataFrame()


def antimalarial_exclusion(analytic: pd.DataFrame | None = None) -> pd.DataFrame:
    """Count afebrile children dropped because recent antimalarial = 1.

    Locked counts: 93,334 afebrile with valid RDT → 138 excluded (0.15%) → 93,196.
    """
    ensure_dirs()
    flow_path = TABLES / "table2_flow.csv"
    if not flow_path.exists():
        raise FileNotFoundError(flow_path)
    flow = pd.read_csv(flow_path)
    s4 = flow.loc[flow["step"] == "4_afebrile_h22eq0"].groupby("country", as_index=False)[["n", "rdt_pos"]].sum()
    s5 = flow.loc[flow["step"] == "5_afebrile_no_recent_antimalarial"].groupby("country", as_index=False)[["n", "rdt_pos"]].sum()
    out = s4.merge(s5, on="country", suffixes=("_pre", "_post"))
    out["n_excluded"] = out["n_pre"] - out["n_post"]
    out["rdt_pos_excluded"] = out["rdt_pos_pre"] - out["rdt_pos_post"]
    total = pd.DataFrame(
        {
            "country": ["TOTAL"],
            "n_pre": [int(out["n_pre"].sum())],
            "rdt_pos_pre": [int(out["rdt_pos_pre"].sum())],
            "n_post": [int(out["n_post"].sum())],
            "rdt_pos_post": [int(out["rdt_pos_post"].sum())],
            "n_excluded": [int(out["n_excluded"].sum())],
            "rdt_pos_excluded": [int(out["rdt_pos_excluded"].sum())],
        }
    )
    tab = pd.concat([out, total], ignore_index=True)
    tab["pct_excluded"] = 100 * tab["n_excluded"] / tab["n_pre"]
    tab.to_csv(TABLES / "antimalarial_exclusion.csv", index=False)
    n_pre, n_ex, n_post = int(total.n_pre.iloc[0]), int(total.n_excluded.iloc[0]), int(total.n_post.iloc[0])
    print(f"Antimalarial exclusion: {n_pre:,} -> {n_ex} excluded ({100 * n_ex / n_pre:.2f}%) -> {n_post:,}")
    return tab


def equal_country_weight_loco(analytic: pd.DataFrame | None = None) -> pd.DataFrame:
    """LOCO with equal total weight per training country. Does not touch primary Table 2."""
    ensure_dirs()
    if analytic is None:
        analytic = pd.read_parquet(DATA_DIR / "analytic.parquet")
    res = run_loco(
        analytic,
        tag="equal_country_weight",
        tune=False,
        equal_country_weight=True,
        reuse_primary_params=True,
        overwrite=True,
        n_boot=200,
        compute_aux=False,
    )
    res["metrics"].to_csv(TABLES / "metrics_loco_equal_country_weight.csv", index=False)
    return res["metrics"]


def _logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _expit(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))


def _intercept_recal(y: np.ndarray, lp: np.ndarray, w: np.ndarray) -> float:
    y = np.asarray(y, dtype=float)
    lp = np.asarray(lp, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(y) & np.isfinite(lp) & (w > 0)
    y, lp, w = y[mask], lp[mask], w[mask]

    def nll(a: float) -> float:
        pn = np.clip(_expit(lp + a), 1e-12, 1 - 1e-12)
        return float(-np.sum(w * (y * np.log(pn) + (1 - y) * np.log(1 - pn))))

    res = minimize_scalar(nll, bounds=(-12, 12), method="bounded")
    return float(res.x)


def _platt_recal(y: np.ndarray, lp: np.ndarray, w: np.ndarray) -> tuple[float, float]:
    y = np.asarray(y, dtype=float)
    lp = np.asarray(lp, dtype=float).reshape(-1, 1)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(y) & np.isfinite(lp.ravel()) & (w > 0)
    if mask.sum() < 20 or len(np.unique(y[mask])) < 2:
        return 1.0, 0.0
    lr = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=500)
    lr.fit(lp[mask], y[mask], sample_weight=w[mask])
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])


def recalibration_sensitivity(preds: pd.DataFrame | None = None) -> pd.DataFrame:
    """Intercept-only and Platt recalibration on a 50/50 split inside each held-out country.

    This is a sensitivity, not a transport test: the held-out country is split after
    LOCO scores are already generated. AUC is nearly unchanged under a monotone map;
    O/E is the quantity that should move.
    """
    ensure_dirs()
    if preds is None:
        path = DATA_DIR / "predictions_primary.parquet"
        preds = pd.read_parquet(path)
    rng = np.random.default_rng(SEED)
    rows = []
    for country, g in preds.groupby("country"):
        g = g.copy()
        if "p_lgb" not in g.columns or (g["rdt"] == 1).sum() < 20:
            continue
        if "psu" in g.columns and g["psu"].nunique() >= 4:
            psus = np.asarray(g["psu"].astype(str).unique())
            rng.shuffle(psus)
            cut = max(1, len(psus) // 2)
            cal_psu = set(psus[:cut])
            cal_mask = g["psu"].astype(str).isin(cal_psu)
        else:
            idx = np.arange(len(g))
            cal_idx, _ = train_test_split(
                idx,
                test_size=0.5,
                random_state=SEED,
                stratify=g["rdt"].to_numpy() if g["rdt"].nunique() == 2 else None,
            )
            cal_mask = np.zeros(len(g), dtype=bool)
            cal_mask[cal_idx] = True
            cal_mask = pd.Series(cal_mask, index=g.index)
        cal = g.loc[cal_mask]
        ev = g.loc[~cal_mask]
        if len(ev) < 30 or (ev["rdt"] == 1).sum() < 5:
            continue
        y_cal = cal["rdt"].to_numpy()
        w_cal = pd.to_numeric(cal["weight"], errors="coerce").fillna(0).to_numpy()
        y_ev = ev["rdt"].to_numpy()
        w_ev = pd.to_numeric(ev["weight"], errors="coerce").fillna(0).to_numpy()
        p0 = ev["p_lgb"].to_numpy()
        lp_cal = _logit(cal["p_lgb"].to_numpy())
        lp_ev = _logit(p0)
        a = _intercept_recal(y_cal, lp_cal, w_cal)
        p_int = _expit(lp_ev + a)
        slope, intercept = _platt_recal(y_cal, lp_cal, w_cal)
        p_platt = _expit(slope * lp_ev + intercept)
        rec = {
            "country": country,
            "n_cal": int(len(cal)),
            "n_eval": int(len(ev)),
            "rdt_pos_eval": int((ev["rdt"] == 1).sum()),
            "auc_pre": weighted_auc(y_ev, p0, w_ev),
            "oe_pre": oe_ratio(y_ev, p0, w_ev),
            "auc_intercept": weighted_auc(y_ev, p_int, w_ev),
            "oe_intercept": oe_ratio(y_ev, p_int, w_ev),
            "intercept_shift": a,
            "auc_platt": weighted_auc(y_ev, p_platt, w_ev),
            "oe_platt": oe_ratio(y_ev, p_platt, w_ev),
            "platt_slope": slope,
            "platt_intercept": intercept,
        }
        rows.append(rec)
    out = pd.DataFrame(rows)
    out.to_csv(TABLES / "sensitivity_recalibration.csv", index=False)
    print(out.to_string(index=False))
    return out


def run_sensitivity() -> None:
    ensure_dirs()
    analytic = pd.read_parquet(DATA_DIR / "analytic.parquet")
    microscopy_sensitivity(analytic)
    hb_sensitivity(analytic)
    drop_incomplete_surveys(analytic)
    fever_dontknow_sensitivity(analytic)


def run_revision_sensitivities(analytic: pd.DataFrame | None = None) -> None:
    """New JPMedAI revision analyses. Does not overwrite primary LOCO tables."""
    ensure_dirs()
    if analytic is None:
        analytic = pd.read_parquet(DATA_DIR / "analytic.parquet")
    antimalarial_exclusion(analytic)
    equal_country_weight_loco(analytic)
    recalibration_sensitivity()


## Inlined `figures_tables.py`


In [ ]:
"""Manuscript tables and 300 DPI figures (plan: 10).

Print layout: two-column journal (~90 mm per panel). Figures 1 and 2 are split
into two 2x2 files so labels stay 8-10 pt instead of an 8-tile grid.
"""

from __future__ import annotations

import shutil
from string import ascii_uppercase

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from matplotlib.patches import FancyBboxPatch


# Ascending weighted RDT prevalence (Table 1).
PREVALENCE_ORDER = [
    "Mauritania",
    "Gambia",
    "Senegal",
    "Niger",
    "Mali",
    "Nigeria",
    "Burkina Faso",
]
DISPLAY_NAME = {"Gambia": "The Gambia"}

# ~90 mm column at print size; 2 panels = two-column page width.
PANEL_W_IN = 90 / 25.4
PANEL_H_IN = 78 / 25.4
GUTTER_IN = 0.22

CALIB_COLOR = "#7F2704"
IDENTITY_COLOR = "#111111"
DCA_NONE = "#4D4D4D"
DCA_ALL = "#111111"
DCA_FEVER = "#014636"
DCA_MODEL = "#7F2704"
WEALTH_COLORS = {
    1: "#08306B",
    2: "#08519C",
    3: "#2171B5",
    4: "#6A51A3",
    5: "#3F007D",
}
URBAN_COLORS = {0: "#084594", 1: "#99000D"}


def _print_style() -> None:
    plt.rcParams.update(
        {
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 10,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
            "legend.fontsize": 8,
            "axes.linewidth": 1.2,
            "axes.edgecolor": "#111111",
            "text.color": "#111111",
            "axes.labelcolor": "#111111",
            "xtick.color": "#111111",
            "ytick.color": "#111111",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.dpi": DPI,
        }
    )


def _letter(ax, i: int) -> None:
    ax.text(
        0.02,
        0.98,
        ascii_uppercase[i],
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
        ha="left",
        color="#000000",
        zorder=5,
    )


def _country_title(country: str, n: int | None = None, n_pos: int | None = None) -> str:
    name = DISPLAY_NAME.get(country, country)
    if n is None:
        return name
    if n_pos is None:
        return f"{name}\nn = {n:,}"
    return f"{name}\nn = {n:,}; RDT+ = {n_pos:,}"


def table1_characteristics(analytic: pd.DataFrame) -> pd.DataFrame:
    df = analytic.loc[analytic["train_sample"]].copy()
    rows = []
    for country, g in df.groupby("country"):
        w = g["weight"].to_numpy()
        rec = {
            "country": country,
            "n": int(len(g)),
            "rdt_pos": int((g["rdt"] == 1).sum()),
            "age_mean": weighted_mean(g["age_months"].to_numpy(), w),
            "pct_female": 100 * weighted_mean(g["female"].fillna(0).to_numpy(), w),
            "pct_urban": 100 * weighted_mean(g["urban"].fillna(0).to_numpy(), w),
            "pct_net_used": 100 * weighted_mean(g["net_used"].fillna(0).to_numpy(), w),
            "wealth_mean": weighted_mean(g["wealth"].to_numpy(), w),
            "weighted_rdt_prev": 100 * weighted_mean(g["rdt"].to_numpy(), w),
        }
        rows.append(rec)
    tab = pd.DataFrame(rows)
    tab.to_csv(TABLES / "table1_characteristics.csv", index=False)
    return tab


def _weighted_pct(series: pd.Series, w: np.ndarray, value=1) -> float:
    s = pd.to_numeric(series, errors="coerce")
    mask = s.notna() & np.isfinite(w) & (w > 0)
    if mask.sum() == 0:
        return float("nan")
    return 100.0 * float(np.average((s[mask] == value).astype(float), weights=w[mask]))


def table1_expanded(analytic: pd.DataFrame) -> pd.DataFrame:
    """Every PRIMARY_FEATURES predictor plus percent missing, by country."""
    df = analytic.loc[analytic["train_sample"]].copy()
    rows = []
    binary = {
        "urban": "Urban residence (%)",
        "water_improved": "Improved water (%)",
        "toilet_improved": "Improved sanitation (%)",
        "net_own": "Household owns a bed-net (%)",
        "net_used": "Slept under a net last night (%)",
        "female": "Female child (%)",
        "sahel_nigeria": "Nigeria Sahel band (%)",
    }
    continuous = {
        "age_months": "Age (months)",
        "wealth": "Wealth quintile (mean, 1=poorest)",
        "educ": "Maternal education code (v106, 0–3)",
        "hhsize": "Household size",
        "month_sin": "Survey month sine",
        "month_cos": "Survey month cosine",
    }
    categorical = {
        "floor_grp": "Floor material",
        "wall_grp": "Wall material",
        "roof_grp": "Roof material",
        "lat_band": "Latitude band",
    }
    for country, g in df.groupby("country"):
        w = pd.to_numeric(g["weight"], errors="coerce").fillna(0).to_numpy()
        rec: dict = {"country": country, "n": int(len(g)), "rdt_pos": int((g["rdt"] == 1).sum())}
        rec["weighted_rdt_prev"] = 100 * weighted_mean(g["rdt"].to_numpy(), w)
        rec["rdt_miss_pct"] = 100 * float(g["rdt"].isna().mean())
        for col, label in continuous.items():
            if col not in g.columns:
                continue
            rec[col] = weighted_mean(pd.to_numeric(g[col], errors="coerce").to_numpy(), w)
            rec[f"{col}_miss_pct"] = 100 * float(pd.to_numeric(g[col], errors="coerce").isna().mean())
            rec[f"{col}_label"] = label
        for col, label in binary.items():
            if col not in g.columns:
                continue
            rec[col] = _weighted_pct(g[col], w, 1)
            rec[f"{col}_miss_pct"] = 100 * float(pd.to_numeric(g[col], errors="coerce").isna().mean())
            rec[f"{col}_label"] = label
        for col, label in categorical.items():
            if col not in g.columns:
                continue
            s = g[col].astype(str).fillna("missing")
            for level in sorted(s.unique()):
                m = s == level
                rec[f"{col}_{level}_pct"] = (
                    100 * float(w[m.to_numpy()].sum() / w.sum()) if w.sum() > 0 else 0.0
                )
            rec[f"{col}_miss_pct"] = rec.get(f"{col}_missing_pct", 100 * float(g[col].isna().mean()))
        for col in PRIMARY_FEATURES:
            if f"{col}_miss_pct" not in rec and col in g.columns:
                rec[f"{col}_miss_pct"] = 100 * float(pd.to_numeric(g[col], errors="coerce").isna().mean())
        rows.append(rec)
    tab = pd.DataFrame(rows)
    tab.to_csv(TABLES / "table1_expanded.csv", index=False)
    long_rows = []
    for r in rows:
        country = r["country"]
        for col in PRIMARY_FEATURES:
            miss = r.get(f"{col}_miss_pct", float("nan"))
            val = r.get(col, float("nan"))
            long_rows.append(
                {
                    "country": country,
                    "predictor": col,
                    "estimate": val,
                    "missing_pct": miss,
                }
            )
    long = pd.DataFrame(long_rows)
    long.to_csv(TABLES / "table1_predictors_long.csv", index=False)
    return tab


def table_s3_subgroups() -> pd.DataFrame:
    """Supplementary Table 3: LightGBM wealth and urban/rural LOCO metrics."""
    path = TABLES / "subgroup_metrics_primary.csv"
    sub = pd.read_csv(path)
    keep = sub[sub["model"] == "lightgbm"].copy()
    keep.to_csv(TABLES / "table_s3_subgroup_metrics.csv", index=False)
    return keep


def figure_s1_workflow() -> None:
    """Study workflow (new supplementary figure; not a redo of Figure 1)."""
    ensure_dirs()
    _print_style()
    fig, ax = plt.subplots(figsize=(7.2, 9.4))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 14)
    ax.axis("off")
    boxes = [
        (7.2, "23 DHS/MIS waves\n7 Sahelian countries"),
        (6.0, "Merge KR child recode with HR\nhousehold RDT (hml35 / sh212)"),
        (4.8, "Eligibility: age 6–59 months;\nvalid RDT; h22 = 0; no recent\nantimalarial (138 excluded, 0.15%)"),
        (3.6, "Predictors obtainable without a\nblood test (PRIMARY_FEATURES)"),
        (2.4, "Inner leave-one-training-country-out\ntuning (LightGBM grid; elastic net fixed)"),
        (1.2, "Outer leave-one-country-out\nevaluation (held-out country unused)"),
        (0.15, "Calibration plots, decision curves,\nSHAP on pooled training LightGBM"),
    ]
    x0, width, height = 1.6, 6.8, 1.05
    for y, text in boxes:
        fancy = FancyBboxPatch(
            (x0, y),
            width,
            height,
            boxstyle="round,pad=0.04,rounding_size=0.12",
            linewidth=1.2,
            edgecolor="#111111",
            facecolor="#F7F7F7",
        )
        ax.add_patch(fancy)
        ax.text(x0 + width / 2, y + height / 2, text, ha="center", va="center", fontsize=9)
    for i in range(len(boxes) - 1):
        y_top = boxes[i][0]
        y_bot = boxes[i + 1][0] + height
        ax.annotate(
            "",
            xy=(5, y_bot + 0.02),
            xytext=(5, y_top - 0.02),
            arrowprops=dict(arrowstyle="-|>", color="#111111", lw=1.4),
        )
    ax.set_title("Figure S1. Model-development and evaluation workflow", fontsize=11, pad=8)
    fig.tight_layout()
    path = FIGURES / "figure_s1_workflow.png"
    fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Wrote {path}")


def write_figure_captions() -> None:

    text = """# Figure captions (revision)

**Figure 1.** Calibration of the primary LightGBM model in leave-one-country-out validation among afebrile children aged 6–59 months with a valid RDT. Each panel is one held-out country, ordered by ascending survey-weighted RDT prevalence. The x-axis is the mean predicted probability in each predicted-risk decile; the y-axis is the survey-weighted observed RDT-positive fraction in that decile. The dashed black line is perfect calibration. Brown points and lines are decile means. O/E > 1 (points above the diagonal) is under-prediction; O/E < 1 is over-prediction. Sample sizes (n, RDT+) are shown in each panel title. Panels A–D: Mauritania (n = 7,063; RDT+ = 54), The Gambia (n = 5,001; RDT+ = 46), Senegal (n = 30,074; RDT+ = 573), Niger (n = 2,737; RDT+ = 410). Panels E–G: Mali (n = 17,033; RDT+ = 3,922), Nigeria (n = 15,979; RDT+ = 5,195), Burkina Faso (n = 15,309; RDT+ = 6,613). Files: `figure1_calibration_a.png`, `figure1_calibration_b.png`. Do not upload the unsplit `figure1_calibration.png`.

**Figure 2.** Decision-curve analysis for the primary LightGBM model on all children aged 6–59 months with a valid RDT and known fever status (not the afebrile subset alone). Each panel is one held-out country, same order as Figure 1. The x-axis is the decision threshold (probability of RDT positivity at which one would choose to test). The y-axis is net benefit. Strategies: dotted grey, test none; dashed black, test all; solid dark green, fever rule (test every child with a two-week fever history); solid brown, fever-plus-model (fever rule, plus testing an afebrile child if predicted probability exceeds the threshold). The light grey band is the pre-specified 5–40% threshold range. Sample sizes in each title are the full 6–59 month RDT sample used for the curves. Panels A–D: Mauritania (n = 8,632; RDT+ = 71), The Gambia (n = 5,949; RDT+ = 57), Senegal (n = 37,148; RDT+ = 876), Niger (n = 4,141; RDT+ = 942). Panels E–G: Mali (n = 22,274; RDT+ = 5,783), Nigeria (n = 24,533; RDT+ = 9,464), Burkina Faso (n = 21,478; RDT+ = 10,268). Files: `figure2_dca_a.png`, `figure2_dca_b.png`. Do not upload the unsplit `figure2_dca.png`.

**Figure 3.** Calibration by DHS wealth quintile using pooled leave-one-country-out LightGBM predictions among afebrile children. The x-axis is mean predicted probability; the y-axis is the survey-weighted observed RDT-positive fraction. The dashed black line is perfect calibration. Quintile 1 is poorest; quintile 5 is richest. Pooled sample: Q1 n = 22,941 (RDT+ = 4,581); Q2 n = 20,890 (4,381); Q3 n = 19,780 (3,968); Q4 n = 16,404 (2,900); Q5 n = 13,181 (983). File: `figure3_wealth_calibration.png`.

**Figure 4.** Decision curves by wealth quintile (A) and urban/rural residence (B) for the primary LightGBM model within the afebrile sample. Because all children in this sample are afebrile, the fever rule is equivalent to test-none; curves show fever-plus-model net benefit. The x-axis is the decision threshold; the y-axis is net benefit. The light grey band is the pre-specified 5–40% range. File: `figure4_subgroup_dca.png`.

**Figure 5.** SHAP attribution for a **pooled training** LightGBM model fitted on all training-sample children with the fixed hyperparameter setting `LGBM_GRID[1]` (`num_leaves=31`, `learning_rate=0.05`, `n_estimators=300`, `max_depth=5`). This is **not** one of the leave-one-country-out models reported in Table 2. TreeExplainer was applied to a random sample of 4,000 pooled training observations (seed 42). Rankings describe how this fitted model used the supplied features; they are not causal effects and they are not attributions from the validated LOCO models. (A) Mean absolute SHAP, highest at the top. (B) Beeswarm: each point is one child; colour is feature value (red = high, blue = low); horizontal position is the contribution to the log-odds of RDT positivity. Files: `figure5_shap_bar.png` (A), `figure5_shap_beeswarm.png` (B).

**Figure S1.** Workflow from survey selection through KR+HR merge, eligibility (including 138 antimalarial exclusions), predictors, inner leave-one-training-country-out tuning, outer leave-one-country-out evaluation, and calibration / decision-curve / SHAP outputs. File: `figure_s1_workflow.png`.
"""
    REPORTS.joinpath("figure_captions.md").write_text(text, encoding="utf-8")
    pack = FIGURES / "submission"
    pack.mkdir(parents=True, exist_ok=True)
    upload = [
        "figure1_calibration_a.png",
        "figure1_calibration_b.png",
        "figure2_dca_a.png",
        "figure2_dca_b.png",
        "figure3_wealth_calibration.png",
        "figure4_subgroup_dca.png",
        "figure5_shap_bar.png",
        "figure5_shap_beeswarm.png",
        "figure_s1_workflow.png",
    ]
    for name in upload:
        src = FIGURES / name
        if src.exists():
            shutil.copyfile(src, pack / name)
    FIGURES.joinpath("FIGURES_USED.md").write_text(
        "\n".join(
            [
                "# Figures to upload (no unsplit duplicates)",
                "",
                "Upload only these files:",
                "",
                "- `figure1_calibration_a.png`",
                "- `figure1_calibration_b.png`",
                "- `figure2_dca_a.png`",
                "- `figure2_dca_b.png`",
                "- `figure3_wealth_calibration.png`",
                "- `figure4_subgroup_dca.png`",
                "- `figure5_shap_bar.png`",
                "- `figure5_shap_beeswarm.png`",
                "- `figure_s1_workflow.png`",
                "",
                "Do **not** upload `figure1_calibration.png`, `figure2_dca.png`, or `figure2_dca_legend.png`.",
                "",
                "Captions: `reports/figure_captions.md`.",
                "",
            ]
        ),
        encoding="utf-8",
    )


def _afebrile_n(tab1: pd.DataFrame | None) -> dict[str, tuple[int, int]]:
    if tab1 is None or tab1.empty:
        tab1 = pd.read_csv(TABLES / "table1_characteristics.csv")
    return {r.country: (int(r.n), int(r.rdt_pos)) for r in tab1.itertuples()}


def _eval_n(analytic: pd.DataFrame | None) -> dict[str, tuple[int, int]]:
    if analytic is not None and "eval_sample" in analytic.columns:
        ev = analytic.loc[analytic["eval_sample"]]
        out = {}
        for country, g in ev.groupby("country"):
            out[str(country)] = (int(len(g)), int((g["rdt"] == 1).sum()))
        return out
    flow = pd.read_csv(TABLES / "table2_flow.csv")
    sub = flow.loc[flow["step"] == "3_fever_item_nonmissing"]
    g = sub.groupby("country", as_index=False).agg(n=("n", "sum"), rdt_pos=("rdt_pos", "sum"))
    return {r.country: (int(r.n), int(r.rdt_pos)) for r in g.itertuples()}


def _ordered_countries(available) -> list[str]:
    have = set(available)
    ordered = [c for c in PREVALENCE_ORDER if c in have]
    extra = sorted(have - set(ordered))
    return ordered + extra


def _save_country_halves(
    countries: list[str],
    plot_fn,
    stem: str,
    n_lookup: dict[str, tuple[int, int]],
    empty_fn=None,
) -> None:
    """Two 2x2 figures (A-D then E-G) so each panel prints at ~90 mm."""
    _print_style()
    chunks = [countries[:4], countries[4:]]
    letter_offset = 0
    for part, chunk in enumerate(chunks, start=1):
        if not chunk:
            continue
        ncols = 2
        nrows = 2
        fig_w = ncols * PANEL_W_IN + GUTTER_IN
        fig_h = nrows * PANEL_H_IN + 0.35
        fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)
        for i in range(nrows * ncols):
            ax = axes[i // ncols][i % ncols]
            if i < len(chunk):
                country = chunk[i]
                plot_fn(ax, country)
                n, n_pos = n_lookup.get(country, (None, None))
                ax.set_title(_country_title(country, n, n_pos), pad=4)
                _letter(ax, letter_offset + i)
            elif empty_fn is not None:
                empty_fn(ax)
            else:
                ax.axis("off")
        fig.tight_layout(w_pad=1.1, h_pad=1.15)
        suffix = "a" if part == 1 else "b"
        path = FIGURES / f"{stem}_{suffix}.png"
        fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        if part == 1:
            shutil.copyfile(path, FIGURES / f"{stem}.png")
        letter_offset += len(chunk)


def figure1_calibration(calib: pd.DataFrame, model: str = "lightgbm", tab1: pd.DataFrame | None = None) -> None:
    sub = calib[calib["model"] == model]
    countries = _ordered_countries(sub["country"].unique())
    n_lookup = _afebrile_n(tab1)

    def plot(ax, country):
        g = sub[sub["country"] == country]
        ax.plot([0, 1], [0, 1], ls="--", color=IDENTITY_COLOR, lw=2.0, zorder=1)
        ax.plot(
            g["pred"],
            g["obs"],
            "o-",
            color=CALIB_COLOR,
            lw=3.0,
            ms=8.5,
            markeredgecolor="#111111",
            markeredgewidth=0.7,
            zorder=2,
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("Predicted probability")
        ax.set_ylabel("Observed RDT-positive fraction")
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])

    def empty(ax):
        ax.axis("off")
        ax.plot([], [], ls="--", color=IDENTITY_COLOR, lw=2.0, label="Perfect calibration")
        ax.plot(
            [],
            [],
            "o-",
            color=CALIB_COLOR,
            lw=3.0,
            ms=8.5,
            markeredgecolor="#111111",
            label="Decile mean (survey-weighted)",
        )
        ax.legend(loc="center", frameon=False, fontsize=8)

    _save_country_halves(countries, plot, "figure1_calibration", n_lookup, empty_fn=empty)


def figure2_dca(
    dca: pd.DataFrame,
    model: str = "lightgbm",
    analytic: pd.DataFrame | None = None,
) -> None:
    sub = dca[dca["model"] == model]
    countries = _ordered_countries(sub["country"].unique())
    n_lookup = _eval_n(analytic)

    def plot(ax, country):
        g = sub[sub["country"] == country].sort_values("threshold")
        ax.axvspan(0.05, 0.40, color="#D9D9D9", alpha=0.18, zorder=0)
        ax.axhline(0, color="#333333", lw=1.2, zorder=1)
        ax.plot(g["threshold"], g["nb_none"], ":", color=DCA_NONE, lw=2.6, label="Test none", zorder=2)
        ax.plot(g["threshold"], g["nb_all"], "--", color=DCA_ALL, lw=2.6, label="Test all", zorder=2)
        ax.plot(g["threshold"], g["nb_fever"], "-", color=DCA_FEVER, lw=3.1, label="Fever rule", zorder=3)
        ax.plot(
            g["threshold"],
            g["nb_fever_plus_model"],
            "-",
            color=DCA_MODEL,
            lw=3.1,
            label="Fever + model",
            zorder=3,
        )
        ax.set_xlabel("Decision threshold")
        ax.set_ylabel("Net benefit")
        ax.set_xlim(0.05, 0.40)

    def empty(ax):
        ax.axis("off")
        handles = [
            Line2D([0], [0], ls=":", color=DCA_NONE, lw=2.6, label="Test none"),
            Line2D([0], [0], ls="--", color=DCA_ALL, lw=2.6, label="Test all"),
            Line2D([0], [0], ls="-", color=DCA_FEVER, lw=3.1, label="Fever rule"),
            Line2D([0], [0], ls="-", color=DCA_MODEL, lw=3.1, label="Fever + model"),
        ]
        ax.legend(handles=handles, loc="center", frameon=False, fontsize=9, title="Strategy")

    _save_country_halves(countries, plot, "figure2_dca", n_lookup, empty_fn=empty)

    _print_style()
    fig, ax = plt.subplots(figsize=(4.2, 2.2))
    ax.plot([], [], ":", color=DCA_NONE, lw=2.2, label="Test none")
    ax.plot([], [], "--", color=DCA_ALL, lw=2.2, label="Test all")
    ax.plot([], [], "-", color=DCA_FEVER, lw=2.5, label="Fever rule")
    ax.plot([], [], "-", color=DCA_MODEL, lw=2.5, label="Fever + model")
    ax.legend(loc="center", frameon=False, fontsize=9)
    ax.axis("off")
    fig.savefig(FIGURES / "figure2_dca_legend.png", dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def figure3_wealth_calibration(preds: pd.DataFrame) -> None:
    if preds.empty or "p_lgb" not in preds.columns:
        return

    _print_style()
    fig_w = 2 * PANEL_W_IN + GUTTER_IN
    fig_h = PANEL_H_IN + 0.4
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.plot([0, 1], [0, 1], ls="--", color=IDENTITY_COLOR, lw=2.0, zorder=1)
    n_by_q = {}
    for w, g in preds.groupby("wealth"):
        if pd.isna(w) or len(g) < 50:
            continue
        q = int(w)
        n_by_q[q] = (int(len(g)), int((g["rdt"] == 1).sum()))
        bins = calibration_bins(g["rdt"], g["p_lgb"], g["weight"])
        if bins.empty:
            continue
        label = f"Q{q}"
        ax.plot(
            bins["pred"],
            bins["obs"],
            "o-",
            color=WEALTH_COLORS.get(q, "#333333"),
            lw=2.8,
            ms=8.0,
            markeredgecolor="#111111",
            markeredgewidth=0.5,
            label=label,
            zorder=2,
        )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed RDT-positive fraction")
    ax.legend(title="Wealth quintile (1 = poorest)", frameon=False, loc="upper left")
    ax.set_title("Pooled leave-one-country-out LightGBM predictions")
    fig.tight_layout()
    fig.savefig(FIGURES / "figure3_wealth_calibration.png", dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    pd.DataFrame(
        [{"wealth_quintile": q, "n": n, "rdt_pos": p} for q, (n, p) in sorted(n_by_q.items())]
    ).to_csv(TABLES / "figure3_wealth_n.csv", index=False)


def figure4_subgroup_dca(analytic: pd.DataFrame, preds: pd.DataFrame) -> None:
    if preds.empty:
        return

    _print_style()
    fig_w = 2 * PANEL_W_IN + GUTTER_IN
    fig_h = PANEL_H_IN + 0.25
    fig, axes = plt.subplots(1, 2, figsize=(fig_w, fig_h))
    merged = preds.copy()
    specs = (
        (axes[0], "wealth", "Wealth quintile", True),
        (axes[1], "urban", "Urban / rural residence", False),
    )
    for ax, col, title, is_wealth in specs:
        ax.axvspan(0.05, 0.40, color="#D9D9D9", alpha=0.18, zorder=0)
        ax.axhline(0, color="#333333", lw=1.2, zorder=1)
        for key, g in merged.groupby(col):
            if pd.isna(key) or (g["rdt"] == 1).sum() < 30:
                continue
            y = g["rdt"].to_numpy()
            w = g["weight"].to_numpy()
            p = g["p_lgb"].to_numpy()
            fever = np.zeros(len(g))
            dca = dca_curves(y, fever, p, w, np.ones(len(g), dtype=bool))
            k = int(key)
            n = len(g)
            n_pos = int((g["rdt"] == 1).sum())
            if is_wealth:
                color = WEALTH_COLORS.get(k, "#333333")
                label = f"Q{k} (n={n:,})"
            else:
                color = URBAN_COLORS.get(k, "#333333")
                label = ("Rural" if k == 0 else "Urban") + f" (n={n:,}; RDT+={n_pos:,})"
            ax.plot(dca["threshold"], dca["nb_fever_plus_model"], color=color, lw=3.0, label=label, zorder=2)
        ax.set_title(title)
        ax.set_xlabel("Decision threshold")
        ax.set_ylabel("Net benefit (afebrile)")
        ax.set_xlim(0.05, 0.40)
        ax.legend(fontsize=7.5, frameon=False, loc="upper right")
    _letter(axes[0], 0)
    _letter(axes[1], 1)
    fig.tight_layout(w_pad=1.2)
    fig.savefig(FIGURES / "figure4_subgroup_dca.png", dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def write_results_markdown(tab1, metrics) -> None:

    lines = [
        "# Results extracts",
        "",
        "## Table 1 (survey-weighted, afebrile training sample)",
        "",
        df_to_md(tab1),
        "",
        "## LOCO discrimination and calibration (primary LightGBM/elastic net)",
        "",
        df_to_md(metrics),
        "",
        "Calibration: `outputs/figures/figure1_calibration_a.png` and `figure1_calibration_b.png`.",
        "DCA: `outputs/figures/figure2_dca_a.png` and `figure2_dca_b.png`.",
        "Standalone captions: `reports/figure_captions.md`.",
        "SHAP is a statement about model behavior, not malaria biology.",
    ]
    REPORTS.joinpath("results_extracts.md").write_text("\n".join(lines), encoding="utf-8")


def run_figures(regenerate_primary: bool = False) -> None:
    """Write Table 1, Table S3, Figure S1, captions. Primary Figures 1–5 stay locked unless asked."""
    ensure_dirs()
    analytic = pd.read_parquet(DATA_DIR / "analytic.parquet")
    tab1 = table1_characteristics(analytic)
    table1_expanded(analytic)
    table_s3_subgroups()
    figure_s1_workflow()
    write_figure_captions()
    pred_path = DATA_DIR / "predictions_primary.parquet"
    preds = pd.read_parquet(pred_path) if pred_path.exists() else pd.DataFrame()
    calib_path = TABLES / "calibration_bins_primary.csv"
    dca_path = TABLES / "dca_primary.csv"
    metrics_path = TABLES / "metrics_loco_primary.csv"
    if regenerate_primary:
        if calib_path.exists():
            figure1_calibration(pd.read_csv(calib_path), tab1=tab1)
        if dca_path.exists():
            figure2_dca(pd.read_csv(dca_path), analytic=analytic)
        if len(preds):
            figure3_wealth_calibration(preds)
            figure4_subgroup_dca(analytic, preds)
    metrics = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
    write_results_markdown(tab1, metrics)
    print("Wrote tables/figures (primary Figures 1-5 locked)" if not regenerate_primary else "Wrote tables/figures")


## Run revision outputs (primary Table 2 locked)

Loads saved primary metrics, then equal-country-weight LOCO, recalibration, expanded Table 1, Table S3, Figure S1.


In [ ]:
ensure_dirs()
primary = pd.read_csv(TABLES / "metrics_loco_primary.csv")
print(primary[primary.model == "lightgbm"][["country", "auc", "oe"]].to_string(index=False))
_ = run_loco(tag="primary", overwrite=False)
analytic = pd.read_parquet(DATA_DIR / "analytic.parquet") if (DATA_DIR / "analytic.parquet").exists() else None
print(antimalarial_exclusion(analytic).tail(1).to_string(index=False))
eq = equal_country_weight_loco(analytic)
print(eq[eq.model == "lightgbm"][["country", "auc", "oe"]].to_string(index=False))
print(recalibration_sensitivity().to_string(index=False))
if analytic is not None:
    table1_expanded(analytic)
table_s3_subgroups()
figure_s1_workflow()
run_figures(regenerate_primary=False)
print("done")
